In [49]:
import warnings
warnings.filterwarnings('ignore')

import os
import gc
import pickle

import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt

from datetime import datetime
from pandas.tseries.offsets import MonthEnd

input_table = 'TRN_DF_ECOM_OFFTAKE_CHAIN_PSKU_NEW'
month_run = '2026-07-31'
os.listdir('/data/aman_singh/acuuracy_check')

['Heuristics_all_combination_qcom_cp_apr_live.xlsx',
 'chek_nan.csv',
 'Heuristics_all_combination_ecom_mar_live.xlsx',
 'prophet_data_train_till_30_Jun_2026 (6).csv',
 'combine_model+missing_forecasts_brand_asm.ipynb',
 'MARICO LIMITED_swiggy_june.xlsx',
 'April-26 Plans.xlsx',
 'QCOM Chain PSKU OTP Output',
 'Heuristics_all_combination_ecom_may_live.xlsx',
 'Heuristics_all_combination_qcom_chain_psku_june_live.xlsx',
 'Norms 202602.csv',
 'Norms 202606.csv',
 'key_check.csv',
 'Norms 202607.csv',
 'all_combination_qcom_july_pred.csv',
 'prophet_data_train_till_30_Jun_2026 (5).csv',
 'Stat Demand Forecast MT_as_on_11th_May_2026.xlsb',
 'acc_framework_may_final.xlsx',
 'acc_offtakes_till_may.csv',
 'swigy_vol_chk.csv',
 'ALL Channels Accuracy_fva.ipynb',
 'Marico Ltd._forecast_Jul 2026_to_Oct 2026.csv',
 'all_combination_ecom_backtest_pred.csv',
 'ecom_chain_psku_offtake_to_secondary_v6_PROD.ipynb',
 'seasonality.xlsx',
 'missing_df_gt_all.csv',
 'SOH - 01 Jun.xlsx',
 'qcom_chain_depot

In [50]:
base_dir = '/data/aman_singh/acuuracy_check'

In [51]:
def list_all_files_in_directory(root):
    out = []

    for path, subdirs, files in os.walk(root):
        for name in files:
            out.append(os.path.join(path, name))

    return out

In [52]:
list_all_files_in_directory(base_dir)

['/data/aman_singh/acuuracy_check/Heuristics_all_combination_qcom_cp_apr_live.xlsx',
 '/data/aman_singh/acuuracy_check/chek_nan.csv',
 '/data/aman_singh/acuuracy_check/Heuristics_all_combination_ecom_mar_live.xlsx',
 '/data/aman_singh/acuuracy_check/prophet_data_train_till_30_Jun_2026 (6).csv',
 '/data/aman_singh/acuuracy_check/combine_model+missing_forecasts_brand_asm.ipynb',
 '/data/aman_singh/acuuracy_check/MARICO LIMITED_swiggy_june.xlsx',
 '/data/aman_singh/acuuracy_check/April-26 Plans.xlsx',
 '/data/aman_singh/acuuracy_check/Heuristics_all_combination_ecom_may_live.xlsx',
 '/data/aman_singh/acuuracy_check/Heuristics_all_combination_qcom_chain_psku_june_live.xlsx',
 '/data/aman_singh/acuuracy_check/Norms 202602.csv',
 '/data/aman_singh/acuuracy_check/Norms 202606.csv',
 '/data/aman_singh/acuuracy_check/key_check.csv',
 '/data/aman_singh/acuuracy_check/Norms 202607.csv',
 '/data/aman_singh/acuuracy_check/all_combination_qcom_july_pred.csv',
 '/data/aman_singh/acuuracy_check/prophe

In [53]:
def discover_channel(file_path):
    # file_path = file_path.split('/')

    # if 'ECOM' in file_path:
    #     return 'ECOM'
    # elif 'QCOM' in file_path:
    #     return 'QCOM'
    # elif 'MT' in file_path:
    #     return 'MT'
    # else:
    #     return 'Channel not found'

    return 'ECOM'


In [54]:
from maricovault.MaricoDB import MaricoSnowflake

def get_dbconnection(db_name):    

    KEY_VAULT_NAME = "prod-pwd"

    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'
    

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection


def read_qtr_ind_rate_table():
    """
    Fetch the club sku information from  DWH_SAP_INDEX_TURNOVER_MONTHWISE table.

    Return:
        qtr_ind_rate_data: pandas dataframe
        - dataframe contains all the results from the index rate table.
    """
    connection = get_dbconnection(db_name='PROD')
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=connection, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    connection.close()
    return qtr_ind_rate


dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


In [55]:
data_query = f"""
    select * from {input_table}
    where month_date >= '2023-01-01' and run_month = '{month_run}'
        
"""

offtake_df = pd.read_sql(data_query, dev_conn)
offtake_df.head()

,MONTH_DATE,PLATFORM_NAME,PARENT_MATERIAL_CODE,VOL_IN_RUM,INDEXBPM,BRAND_CODE,IMPUTED,BIG_BILLION_DAYS,BIG_BILLION_DAYS_LAG_1,BIG_BILLION_DAYS_LAG_2,BIG_BILLION_DAYS_LEAD_1,BIG_BILLION_DAYS_LEAD_2,GREAT_INDIAN_FESTIVAL,GREAT_INDIAN_FESTIVAL_LAG_1,GREAT_INDIAN_FESTIVAL_LAG_2,GREAT_INDIAN_FESTIVAL_LEAD_1,GREAT_INDIAN_FESTIVAL_LEAD_2,RATIO_LAST_YEAR,QUARTER,RUN_MONTH
0,2023-01-31,Amazon ARIPL,718288,9.640,13.27071,SAFF GOLD,0,0,0,0,0,0,0,0,0,0,0,0.994832,1,2026-07-31
1,2023-02-28,Amazon ARIPL,718288,7.915,10.89602,SAFF GOLD,0,0,0,0,0,0,0,0,0,0,0,0.861631,1,2026-07-31
2,2023-03-31,Amazon ARIPL,718288,9.505,13.08486,SAFF GOLD,0,0,0,0,0,0,0,0,0,0,0,1.177605,1,2026-07-31
3,2023-04-30,Amazon ARIPL,718288,9.290,12.78889,SAFF GOLD,0,0,0,0,0,0,0,0,0,0,0,0.834181,2,2026-07-31
4,2023-05-31,Amazon ARIPL,718288,8.050,11.08187,SAFF GOLD,0,0,0,0,0,0,0,0,0,0,0,0.957359,2,2026-07-31


In [56]:
offtake_df.columns = offtake_df.columns.str.lower()

In [57]:
offtake_df.duplicated(
    subset=['platform_name','parent_material_code', 'month_date']).sum()

0

In [58]:
offtake_df['brand_code'] = np.where(
    ((offtake_df['parent_material_code'] == 715096) &
    (offtake_df['brand_code'] == 'CO_SO_PCP')),
    'CO_SO_FS',
    offtake_df['brand_code']
)

In [59]:
# offtake_df = offtake_df[offtake_df['platform_name'].isin(
#     ['Amazon', 'Big Basket', 'Flipkart Grocery', 'Flipkart National'])]

In [60]:
offtake_df['key'] = offtake_df[['platform_name','parent_material_code']].astype(str).agg('_'.join, axis=1)
# offtake_df.rename(columns={'realigned_psku': 'parent_material_code'}, inplace=True)
# offtake_df.drop([ 'run_month'], axis=1, inplace=True)
offtake_df['parent_material_code'] = offtake_df['parent_material_code'].astype(int)

In [61]:
offtake_df.duplicated(subset=['key', 'month_date']).sum()

0

In [62]:
(offtake_df['key'] == offtake_df[['platform_name','parent_material_code']].astype(str).agg('_'.join, axis=1)).all()

True

In [63]:
# realigned_df.to_csv('OT_data_debug.csv', index=False)

### Collate MIL

In [64]:
base_dir

'/data/aman_singh/acuuracy_check'

In [65]:
def collate_file(file_hint, extension='.csv'):
    collated_file = pd.DataFrame()

    run_path = f'{base_dir}'
    all_files = list_all_files_in_directory(run_path)

    for file_path in all_files:
        if file_hint in file_path:
            if extension == '.csv':
                print(file_path)
                read_file = pd.read_csv(file_path)
                # read_file['channel'] = discover_channel(file_path)
                read_file['run'] = 'run'
                read_file['step'] = file_path.split('/')[3]
                read_file['file_path'] = file_path

                collated_file = pd.concat(
                    [collated_file, read_file]
                )
                del read_file

    return collated_file

In [66]:
trend_file_df = collate_file('trend_file_train_till')
prophet_file_df = collate_file('prophet_data_train_till')

/data/aman_singh/acuuracy_check/trend_file_train_till_30_Jun_2026 (5).csv
/data/aman_singh/acuuracy_check/trend_file_train_till_30_Jun_2026 (6).csv
/data/aman_singh/acuuracy_check/prophet_data_train_till_30_Jun_2026 (6).csv
/data/aman_singh/acuuracy_check/prophet_data_train_till_30_Jun_2026 (5).csv


In [67]:
# forecast_train_till_file_df = collate_file('forecast_train_till_')

In [68]:
# forecast_train_till_file_df

In [69]:
trend_file_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,...,train_till,cov,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2
0,Big Basket_715100,2023-01-31,0.000000,0.500000,0.683333,0.307375,0.398889,0.000000,0.000061,0.000083,...,2026-06-30,2.049207,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN
1,Big Basket_715100,2023-02-28,0.400000,0.500000,0.683333,0.470485,0.416667,0.000049,0.000061,0.000083,...,2026-06-30,2.049207,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN
2,Big Basket_715100,2023-03-31,0.400000,0.500000,0.683333,0.378150,0.636667,0.000049,0.000061,0.000083,...,2026-06-30,2.049207,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN
3,Big Basket_715100,2023-04-30,0.516732,0.500000,0.683333,0.527226,0.672222,0.000063,0.000061,0.000083,...,2026-06-30,2.049207,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN
4,Big Basket_715100,2023-05-31,0.577807,0.600000,0.683333,0.492230,0.953333,0.000070,0.000073,0.000083,...,2026-06-30,2.049207,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26422,Amazon RK_810674,2026-10-31,53.891363,23.291333,21.067667,45.014039,21.061021,0.069308,0.029954,0.027094,...,2026-06-30,0.312607,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0
26423,Amazon RK_810674,2026-11-30,60.031585,23.291333,21.067667,41.289362,22.476374,0.077204,0.029954,0.027094,...,2026-06-30,0.312607,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0
26424,Amazon RK_810674,2026-12-31,66.249983,23.291333,21.067667,45.573890,22.709541,0.085202,0.029954,0.027094,...,2026-06-30,0.312607,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0
26425,Amazon RK_810674,2027-01-31,72.427165,23.291333,21.067667,55.934384,21.755724,0.093146,0.029954,0.027094,...,2026-06-30,0.312607,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0


In [70]:
trend_file_df.columns

Index(['key', 'month_date', 'pred_SARIMA', 'pred_p3m', 'pred_p6m',
       'pred_prophet', 'pred_rf', 'pred_value_SARIMA', 'pred_value_p3m',
       'pred_value_p6m', 'pred_value_prophet', 'pred_value_rf',
       'parent_material_code', 'platform_name', 'vol_in_rum', 'brand_code',
       'big_billion_days', 'big_billion_days_lag_1', 'big_billion_days_lag_2',
       'big_billion_days_lead_1', 'big_billion_days_lead_2', 'ratio_last_year',
       'quarter', 'qtr_ind_rate', 'vol_in_rum_value', 'vol_in_rum_treated',
       'vol_in_rum_value_treated', 'train_till', 'cov', 'run', 'step',
       'file_path', 'great_indian_festival', 'great_indian_festival_lag_1',
       'great_indian_festival_lag_2', 'great_indian_festival_lead_1',
       'great_indian_festival_lead_2'],
      dtype='object')

In [71]:
trend_file_df['platform_name'].unique()

array(['Big Basket', 'Flipkart Grocery', 'Flipkart National', 'Meesho',
       'Myntra', 'Nykaa', 'Amazon ARIPL', 'Amazon RK'], dtype=object)

In [72]:
trend_file_df['month_date'] = pd.to_datetime(trend_file_df['month_date'])
prophet_file_df['month_date'] = pd.to_datetime(prophet_file_df['month_date'])

trend_file_df['train_till'] = pd.to_datetime(trend_file_df['train_till'])
prophet_file_df['train_till'] = pd.to_datetime(prophet_file_df['train_till'])

trend_file_df['run_month'] = pd.to_datetime(trend_file_df['train_till'] + MonthEnd(1))
prophet_file_df['run_month'] = pd.to_datetime(prophet_file_df['train_till'] + MonthEnd(1))

In [73]:
trend_file_df.duplicated(subset=['key', 'month_date', 'run_month']).sum(), \
prophet_file_df.duplicated(subset=['key', 'month_date', 'run_month']).sum()

(0, 0)

In [74]:
mappings = {}

for run_month in trend_file_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   

{Timestamp('2026-07-31 00:00:00'): {Timestamp('2026-07-31 00:00:00'): 'M',
  Timestamp('2026-08-31 00:00:00'): 'M+1',
  Timestamp('2026-09-30 00:00:00'): 'M+2',
  Timestamp('2026-10-31 00:00:00'): 'M+3',
  Timestamp('2026-11-30 00:00:00'): 'M+4',
  Timestamp('2026-12-31 00:00:00'): 'M+5',
  Timestamp('2027-01-31 00:00:00'): 'M+6',
  Timestamp('2027-02-28 00:00:00'): 'M+7',
  Timestamp('2027-03-31 00:00:00'): 'M+8'}}

In [75]:
trend_file_df['M month'] = trend_file_df.apply(
    lambda x: mappings[x['run_month']].get(
        x['month_date']
    ), axis=1
)

In [76]:
trend_file_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,...,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,run_month,M month
0,Big Basket_715100,2023-01-31,0.000000,0.500000,0.683333,0.307375,0.398889,0.000000,0.000061,0.000083,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-07-31,None
1,Big Basket_715100,2023-02-28,0.400000,0.500000,0.683333,0.470485,0.416667,0.000049,0.000061,0.000083,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-07-31,None
2,Big Basket_715100,2023-03-31,0.400000,0.500000,0.683333,0.378150,0.636667,0.000049,0.000061,0.000083,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-07-31,None
3,Big Basket_715100,2023-04-30,0.516732,0.500000,0.683333,0.527226,0.672222,0.000063,0.000061,0.000083,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-07-31,None
4,Big Basket_715100,2023-05-31,0.577807,0.600000,0.683333,0.492230,0.953333,0.000070,0.000073,0.000083,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-07-31,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26422,Amazon RK_810674,2026-10-31,53.891363,23.291333,21.067667,45.014039,21.061021,0.069308,0.029954,0.027094,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,M+3
26423,Amazon RK_810674,2026-11-30,60.031585,23.291333,21.067667,41.289362,22.476374,0.077204,0.029954,0.027094,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,M+4
26424,Amazon RK_810674,2026-12-31,66.249983,23.291333,21.067667,45.573890,22.709541,0.085202,0.029954,0.027094,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,M+5
26425,Amazon RK_810674,2027-01-31,72.427165,23.291333,21.067667,55.934384,21.755724,0.093146,0.029954,0.027094,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,M+6


In [77]:
trend_file_df[trend_file_df['M month'].notna()]

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,...,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,run_month,M month
42,Big Basket_715100,2026-07-31,0.000628,0.000000,0.000000,0.000000,0.011111,7.661616e-08,0.000000,0.000000,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-07-31,M
43,Big Basket_715100,2026-08-31,0.000628,0.000000,0.000000,0.000000,0.000000,7.661616e-08,0.000000,0.000000,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-07-31,M+1
44,Big Basket_715100,2026-09-30,0.000628,0.000000,0.000000,0.000000,0.272222,7.661616e-08,0.000000,0.000000,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-07-31,M+2
45,Big Basket_715100,2026-10-31,0.000628,0.000000,0.000000,0.000000,0.061111,7.661616e-08,0.000000,0.000000,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-07-31,M+3
46,Big Basket_715100,2026-11-30,0.000628,0.000000,0.000000,0.000000,0.010000,7.661616e-08,0.000000,0.000000,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-07-31,M+4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26422,Amazon RK_810674,2026-10-31,53.891363,23.291333,21.067667,45.014039,21.061021,6.930769e-02,0.029954,0.027094,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,M+3
26423,Amazon RK_810674,2026-11-30,60.031585,23.291333,21.067667,41.289362,22.476374,7.720441e-02,0.029954,0.027094,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,M+4
26424,Amazon RK_810674,2026-12-31,66.249983,23.291333,21.067667,45.573890,22.709541,8.520166e-02,0.029954,0.027094,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,M+5
26425,Amazon RK_810674,2027-01-31,72.427165,23.291333,21.067667,55.934384,21.755724,9.314591e-02,0.029954,0.027094,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,M+6


In [78]:
trend_file_df[['run_month', 'train_till']].drop_duplicates().sort_values(by=['run_month'])

,run_month,train_till
0,2026-07-31,2026-06-30


In [79]:
prophet_file_df[['run_month', 'train_till']].drop_duplicates().sort_values(by=['run_month'])

,run_month,train_till
0,2026-07-31,2026-06-30


In [80]:
trend_file_df['M month'].unique()

array([None, 'M', 'M+1', 'M+2', 'M+3', 'M+4', 'M+5', 'M+6', 'M+7'],
      dtype=object)

In [81]:
brand_md_df = pd.read_excel(r"/data/aman_singh/mt_forecast/Brand_metadata.xlsx")

In [82]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    brand_md_df,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(trend_file_df)

In [83]:
trend_file_df['portfolio'].isna().sum()

0

In [84]:
prophet_file_df[
    ['month_date', 'key', 'run_month']
].duplicated().sum()

0

In [85]:
prophet_file_df

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,yhat_60_%ile,yhat_70_%ile,yhat_75_%ile,trend_60_%ile,...,big_billion_days_lag_2,big_billion_days_lag_2_lower,big_billion_days_lag_2_upper,big_billion_days_lead_1,big_billion_days_lead_1_lower,big_billion_days_lead_1_upper,big_billion_days_lead_2,big_billion_days_lead_2_lower,big_billion_days_lead_2_upper,run_month
0,2023-01-31,7.432016,5.476805,12.854018,7.432016,7.432016,9.840139,10.583375,10.953920,7.432016,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-31
1,2023-02-28,7.484860,3.775993,11.272572,7.484860,7.484860,8.484011,9.131529,9.484775,7.484860,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-31
2,2023-03-31,7.543366,8.406729,15.590738,7.543366,7.543366,12.732233,13.541509,13.897051,7.543366,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-31
3,2023-04-30,7.599984,1.805063,8.959324,7.599984,7.599984,6.069386,6.861764,7.255116,7.599984,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-31
4,2023-05-31,7.658490,5.014820,12.233505,7.658490,7.658490,9.501385,10.196926,10.583671,7.658490,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67000,2026-10-31,2.618192,2.490825,2.490825,2.618192,2.618192,2.490825,2.490825,2.490825,2.618192,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-07-31
67001,2026-11-30,2.745891,2.802472,2.802472,2.745891,2.745891,2.802472,2.802472,2.802472,2.745891,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-07-31
67002,2026-12-31,2.877848,3.197872,3.197872,2.877847,2.877848,3.197872,3.197872,3.197872,2.877848,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-07-31
67003,2027-01-31,3.009804,3.135972,3.135972,3.009804,3.009804,3.135972,3.135972,3.135972,3.009804,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-07-31


In [86]:
# Merge 70th percentile Prophet predictions
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    prophet_file_df[['month_date', 'key', 'run_month', 'yhat_70_%ile','yhat_60_%ile']].rename(
        columns={
            'yhat_70_%ile': 'pred_prophet_70%ile',
            'yhat_60_%ile': 'pred_prophet_60%ile'
        }
    ),
    on=['month_date', 'key', 'run_month'],
    how='left'
)
assert len(trend_file_df) == len_before_merge

In [87]:
assert trend_file_df.duplicated(
    subset=['run_month', 'month_date', 'key']
).sum() == 0

In [88]:
trend_file_df.drop('vol_in_rum', axis=1, inplace=True)

In [89]:
offtake_df.head()

,month_date,platform_name,parent_material_code,vol_in_rum,indexbpm,brand_code,imputed,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,...,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,run_month,key
0,2023-01-31,Amazon ARIPL,718288,9.640,13.27071,SAFF GOLD,0,0,0,0,...,0,0,0,0,0,0,0.994832,1,2026-07-31,Amazon ARIPL_718288
1,2023-02-28,Amazon ARIPL,718288,7.915,10.89602,SAFF GOLD,0,0,0,0,...,0,0,0,0,0,0,0.861631,1,2026-07-31,Amazon ARIPL_718288
2,2023-03-31,Amazon ARIPL,718288,9.505,13.08486,SAFF GOLD,0,0,0,0,...,0,0,0,0,0,0,1.177605,1,2026-07-31,Amazon ARIPL_718288
3,2023-04-30,Amazon ARIPL,718288,9.290,12.78889,SAFF GOLD,0,0,0,0,...,0,0,0,0,0,0,0.834181,2,2026-07-31,Amazon ARIPL_718288
4,2023-05-31,Amazon ARIPL,718288,8.050,11.08187,SAFF GOLD,0,0,0,0,...,0,0,0,0,0,0,0.957359,2,2026-07-31,Amazon ARIPL_718288


In [90]:
assert offtake_df.duplicated(subset=['key', 'month_date']).sum() == 0

In [91]:
offtake_df['month_date'] = pd.to_datetime(offtake_df['month_date'])
offtake_df['run_month'] = pd.to_datetime(offtake_df['run_month'])


In [92]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    offtake_df[['key', 'run_month','month_date', 'vol_in_rum']],
    on=['run_month','month_date', 'key'],
    how='left'
)
assert len(trend_file_df) == len_before_merge

In [93]:
trend_file_df.select_dtypes('number').isna().sum()

pred_SARIMA                       364
pred_p3m                            0
pred_p6m                            0
pred_prophet                    26188
pred_rf                             0
pred_value_SARIMA                 364
pred_value_p3m                      0
pred_value_p6m                      0
pred_value_prophet              26188
pred_value_rf                       0
parent_material_code                0
big_billion_days                26427
big_billion_days_lag_1          26427
big_billion_days_lag_2          26427
big_billion_days_lead_1         26427
big_billion_days_lead_2         26427
ratio_last_year                 10018
quarter                             0
qtr_ind_rate                        0
vol_in_rum_value                    0
vol_in_rum_treated                  0
vol_in_rum_value_treated            0
cov                                 0
great_indian_festival           88124
great_indian_festival_lag_1     88124
great_indian_festival_lag_2     88124
great_indian

In [94]:
trend_file_df.select_dtypes('number').min().round()

pred_SARIMA                     -21186.0
pred_p3m                             0.0
pred_p6m                             0.0
pred_prophet                         0.0
pred_rf                              0.0
pred_value_SARIMA                   -1.0
pred_value_p3m                       0.0
pred_value_p6m                       0.0
pred_value_prophet                   0.0
pred_value_rf                        0.0
parent_material_code            709538.0
big_billion_days                     0.0
big_billion_days_lag_1               0.0
big_billion_days_lag_2               0.0
big_billion_days_lead_1              0.0
big_billion_days_lead_2              0.0
ratio_last_year                      0.0
quarter                              1.0
qtr_ind_rate                         0.0
vol_in_rum_value                     0.0
vol_in_rum_treated                   0.0
vol_in_rum_value_treated             0.0
cov                                  0.0
great_indian_festival                0.0
great_indian_fes

In [95]:
trend_file_df['vol_in_rum'].fillna(0, inplace=True)

In [96]:
for col in [ 'pred_prophet_70%ile','pred_prophet_60%ile', 'vol_in_rum']:
    trend_file_df[col] = trend_file_df[col].clip(lower=0)

In [97]:
trend_file_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,...,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum
0,Big Basket_715100,2023-01-31,0.000000,0.500000,0.683333,0.307375,0.398889,0.000000,0.000061,0.000083,...,NaN,NaN,NaN,NaN,2026-07-31,None,Skin Care,0.416286,0.356045,0.4
1,Big Basket_715100,2023-02-28,0.400000,0.500000,0.683333,0.470485,0.416667,0.000049,0.000061,0.000083,...,NaN,NaN,NaN,NaN,2026-07-31,None,Skin Care,0.563807,0.514144,0.4
2,Big Basket_715100,2023-03-31,0.400000,0.500000,0.683333,0.378150,0.636667,0.000049,0.000061,0.000083,...,NaN,NaN,NaN,NaN,2026-07-31,None,Skin Care,0.481039,0.421126,0.7
3,Big Basket_715100,2023-04-30,0.516732,0.500000,0.683333,0.527226,0.672222,0.000063,0.000061,0.000083,...,NaN,NaN,NaN,NaN,2026-07-31,None,Skin Care,0.647293,0.593691,0.7
4,Big Basket_715100,2023-05-31,0.577807,0.600000,0.683333,0.492230,0.953333,0.000070,0.000073,0.000083,...,NaN,NaN,NaN,NaN,2026-07-31,None,Skin Care,0.597152,0.542531,1.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114546,Amazon RK_810674,2026-10-31,53.891363,23.291333,21.067667,45.014039,21.061021,0.069308,0.029954,0.027094,...,0.0,0.0,0.0,0.0,2026-07-31,M+3,Hair Oils,45.989012,45.318774,0.0
114547,Amazon RK_810674,2026-11-30,60.031585,23.291333,21.067667,41.289362,22.476374,0.077204,0.029954,0.027094,...,0.0,0.0,0.0,0.0,2026-07-31,M+4,Hair Oils,42.960602,41.802791,0.0
114548,Amazon RK_810674,2026-12-31,66.249983,23.291333,21.067667,45.573890,22.709541,0.085202,0.029954,0.027094,...,0.0,0.0,0.0,0.0,2026-07-31,M+5,Hair Oils,48.057293,46.489907,0.0
114549,Amazon RK_810674,2027-01-31,72.427165,23.291333,21.067667,55.934384,21.755724,0.093146,0.029954,0.027094,...,0.0,0.0,0.0,0.0,2026-07-31,M+6,Hair Oils,59.460121,57.502119,0.0


In [98]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)

In [99]:
trend_file_df['P3M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=3, min_periods=3).mean()

trend_file_df['P6M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=6, min_periods=6).mean()

trend_file_df['LY P3M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=3, min_periods=3).mean()

trend_file_df['LY P6M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=6, min_periods=6).mean()

In [100]:
trend_file_df['LY P3M_copy'] = trend_file_df['LY P3M'].copy()

In [101]:
trend_file_df['P3M Max'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).max()

In [102]:
trend_file_df['P3M Top 2 Mean'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sort(x)[-2:].mean(), raw=True) 

# lambda x: x.nlargest(2).mean(), raw=False

In [103]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)

In [104]:
trend_file_df['MoM P3M growth'] = (
    trend_file_df.groupby(['run_month', 'key'])['P3M']
      .pct_change() * 100
)

In [105]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
trend_file_df['MoM P3M growth_lag_1'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(1)
trend_file_df['MoM P3M growth_lag_2'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(2)



In [106]:
trend_file_df['>=20%_3M_inc_month_count'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['MoM P3M growth'].shift(0)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sum(x >= 20), raw=True) 

In [107]:
trend_file_df['Avg(P3M Mean, Max)'] = trend_file_df[['P3M', 'P3M Max']].mean(axis=1)

In [108]:
for col in ['P3M', 'P6M', 'LY P3M', 'P3M Max', 'P3M Top 2 Mean', 
            'MoM P3M growth', '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)',
            'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2' ]:
    # if not 'LY' in col:  'LY P6M',
    trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
    trend_file_df[col] = trend_file_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )


# for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
#     if not 'LY' in col:
#         trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
#         trend_file_df[col] = trend_file_df.groupby(['key'], as_index = True, group_keys = False)[col].apply(
#             lambda x: x.ffill()
#         )

In [109]:
for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
    trend_file_df[f'{col}_value'] = trend_file_df[col] * trend_file_df['qtr_ind_rate'] / (10 ** 7)

In [110]:
trend_file_df['vol_in_rum_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['vol_in_rum'] / (10 ** 7)
trend_file_df['pred_prophet_70%ile_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['pred_prophet_70%ile'] / (10 ** 7)
trend_file_df['pred_prophet_60%ile_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['pred_prophet_60%ile'] / (10 ** 7)

In [111]:
value_cols = [col for col in trend_file_df.columns if 'value' in col]
value_cols

['pred_value_SARIMA',
 'pred_value_p3m',
 'pred_value_p6m',
 'pred_value_prophet',
 'pred_value_rf',
 'vol_in_rum_value',
 'vol_in_rum_value_treated',
 'P3M_value',
 'P6M_value',
 'LY P3M_value',
 'LY P6M_value',
 'pred_prophet_70%ile_value',
 'pred_prophet_60%ile_value']

In [112]:
for col in value_cols:
    try:
        assert trend_file_df[col].min() >= 0
    except:
        print(col)
    

    # trend_file_df[col] = trend_file_df[col] / (10 ** 7)

pred_value_SARIMA


In [113]:
trend_file_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,...,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value
98404,Amazon RK_718472,2023-11-30,0.000000e+00,0.51,1.125,NaN,0.487286,0.000000e+00,0.000025,0.000056,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98405,Amazon RK_718472,2023-12-31,4.499998e-01,0.51,1.125,NaN,0.470571,2.235727e-05,0.000025,0.000056,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98406,Amazon RK_718472,2024-01-31,1.431689e-01,0.51,1.125,NaN,1.137857,7.113037e-06,0.000025,0.000056,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98407,Amazon RK_718472,2024-02-29,7.860552e-01,0.51,1.125,NaN,1.496571,3.905346e-05,0.000025,0.000056,...,NaN,NaN,NaN,0.51,0.000025,NaN,NaN,NaN,NaN,NaN
98408,Amazon RK_718472,2024-03-31,7.437929e-01,0.90,1.125,NaN,2.247429,3.695375e-05,0.000045,0.000056,...,NaN,NaN,NaN,0.90,0.000045,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7246,Big Basket_719192,2026-10-31,-1.258270e-309,0.00,0.000,NaN,0.000000,-1.258270e-314,0.000000,0.000000,...,-100.0,-100.0,0.0,0.00,0.000000,0.0,0.0,0.0,NaN,NaN
7247,Big Basket_719192,2026-11-30,-5.203838e-315,0.00,0.000,NaN,0.000000,-5.203993e-320,0.000000,0.000000,...,-100.0,-100.0,0.0,0.00,0.000000,0.0,0.0,0.0,NaN,NaN
7248,Big Basket_719192,2026-12-31,-1.196875e-309,0.00,0.000,NaN,0.000000,-1.196875e-314,0.000000,0.000000,...,-100.0,-100.0,0.0,0.00,0.000000,0.0,0.0,0.0,NaN,NaN
7249,Big Basket_719192,2027-01-31,-5.250576e-315,0.00,0.000,NaN,0.000000,-5.250436e-320,0.000000,0.000000,...,-100.0,-100.0,0.0,0.00,0.000000,0.0,0.0,0.0,NaN,NaN


In [114]:
assert trend_file_df.duplicated(
    subset=['run_month', 'brand_code', 'key', 'month_date']
).sum() == 0

In [115]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
trend_file_df['LY'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(12)

trend_file_df['LLY'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(24)


trend_file_df['LY value'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(12)

trend_file_df['LLY value'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(24)


trend_file_df['OT_Value_in_Cr_lag_1'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(1)

trend_file_df['OT_Value_in_Cr_lag_2'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(2)

trend_file_df['OT_Value_in_Cr_lag_3'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(3)

In [116]:
for col in ['OT_Value_in_Cr_lag_1', 'OT_Value_in_Cr_lag_2', 'OT_Value_in_Cr_lag_3']:
    # if not 'LY' in col:  'LY P6M',
    trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
    trend_file_df[col] = trend_file_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

In [117]:
trend_file_df.columns

Index(['key', 'month_date', 'pred_SARIMA', 'pred_p3m', 'pred_p6m',
       'pred_prophet', 'pred_rf', 'pred_value_SARIMA', 'pred_value_p3m',
       'pred_value_p6m', 'pred_value_prophet', 'pred_value_rf',
       'parent_material_code', 'platform_name', 'brand_code',
       'big_billion_days', 'big_billion_days_lag_1', 'big_billion_days_lag_2',
       'big_billion_days_lead_1', 'big_billion_days_lead_2', 'ratio_last_year',
       'quarter', 'qtr_ind_rate', 'vol_in_rum_value', 'vol_in_rum_treated',
       'vol_in_rum_value_treated', 'train_till', 'cov', 'run', 'step',
       'file_path', 'great_indian_festival', 'great_indian_festival_lag_1',
       'great_indian_festival_lag_2', 'great_indian_festival_lead_1',
       'great_indian_festival_lead_2', 'run_month', 'M month', 'portfolio',
       'pred_prophet_70%ile', 'pred_prophet_60%ile', 'vol_in_rum', 'P3M',
       'P6M', 'LY P3M', 'LY P6M', 'LY P3M_copy', 'P3M Max', 'P3M Top 2 Mean',
       'MoM P3M growth', 'MoM P3M growth_lag_1', 'MoM 

In [118]:
# trend_file_df[['ASM', 'Depot', 'PSKU']] = trend_file_df['key'].str.split('_', expand=True)

In [119]:
trend_file_df.reset_index(drop=True, inplace=True)

In [120]:
trend_file_df.shape

(114551, 67)

In [121]:
trend_file_df['key'].nunique()

2686

In [122]:
# batch_info = pd.read_excel(
#     '/data/aniket/az_demand_forecasting-mil-sc/channel_wise_batch.xlsx'
# )

In [123]:
# brand_class_df = pd.read_excel(r"/data/aman_singh/acuuracy_check/Brand_Classification.xlsb")
# brand_class_df.head()

In [124]:
# brand_class_df.columns = ['brand_code', 'class']


In [125]:
# len_before_merge = len(trend_file_df)
# trend_file_df = trend_file_df.merge(
#     brand_class_df, 
#     on=['brand_code'],
#     how='left'
# )
# assert len_before_merge == len(trend_file_df)
# del len_before_merge

In [126]:
# trend_file_df['class'].isna().sum()

In [127]:
# trend_file_df['class'].unique()

missing combinations

In [128]:
# model_file = pd.read_csv("/data/aman_singh/acuuracy_check/Heuristic_QCOM_Chain_PSKU_Offtakes_live2.csv")
# model_file

In [129]:
model_file = trend_file_df.copy()

In [130]:
model_file['key'].nunique()

2686

In [131]:
run_month

Timestamp('2026-07-31 00:00:00')

In [132]:
data_query = f"""
    select * from {input_table}
    where month_date >= '2023-01-01' and run_month = '{month_run}'
        
"""
qcom_df = pd.read_sql(data_query, dev_conn)
qcom_df.head()

,MONTH_DATE,PLATFORM_NAME,PARENT_MATERIAL_CODE,VOL_IN_RUM,INDEXBPM,BRAND_CODE,IMPUTED,BIG_BILLION_DAYS,BIG_BILLION_DAYS_LAG_1,BIG_BILLION_DAYS_LAG_2,BIG_BILLION_DAYS_LEAD_1,BIG_BILLION_DAYS_LEAD_2,GREAT_INDIAN_FESTIVAL,GREAT_INDIAN_FESTIVAL_LAG_1,GREAT_INDIAN_FESTIVAL_LAG_2,GREAT_INDIAN_FESTIVAL_LEAD_1,GREAT_INDIAN_FESTIVAL_LEAD_2,RATIO_LAST_YEAR,QUARTER,RUN_MONTH
0,2023-01-31,Amazon ARIPL,718288,9.640,13.27071,SAFF GOLD,0,0,0,0,0,0,0,0,0,0,0,0.994832,1,2026-07-31
1,2023-02-28,Amazon ARIPL,718288,7.915,10.89602,SAFF GOLD,0,0,0,0,0,0,0,0,0,0,0,0.861631,1,2026-07-31
2,2023-03-31,Amazon ARIPL,718288,9.505,13.08486,SAFF GOLD,0,0,0,0,0,0,0,0,0,0,0,1.177605,1,2026-07-31
3,2023-04-30,Amazon ARIPL,718288,9.290,12.78889,SAFF GOLD,0,0,0,0,0,0,0,0,0,0,0,0.834181,2,2026-07-31
4,2023-05-31,Amazon ARIPL,718288,8.050,11.08187,SAFF GOLD,0,0,0,0,0,0,0,0,0,0,0,0.957359,2,2026-07-31


In [133]:
qcom_df.columns = qcom_df.columns.str.lower()

In [134]:
qcom_df['key'] = qcom_df[['platform_name', 'parent_material_code']].astype(str).agg('_'.join, axis=1)

In [135]:
model_file['run_month'] = pd.to_datetime(model_file['run_month'])
model_file['month_date'] = pd.to_datetime(model_file['month_date'])

qcom_df['run_month'] = pd.to_datetime(qcom_df['run_month'])
qcom_df['month_date'] = pd.to_datetime(qcom_df['month_date'])

In [136]:
tmp_df = model_file.groupby(['key', 'run_month'])['LY'].count().reset_index()
tmp_df#.isnull().sum()
#qcom_df[~qcom_df['key'].isin(model_file['key'].unique())]
qcom_df = qcom_df.merge(tmp_df, on = ['key', 'run_month'], how = 'left')
missing_df = qcom_df[qcom_df['LY'].isna()]
missing_df


,month_date,platform_name,parent_material_code,vol_in_rum,indexbpm,brand_code,imputed,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,...,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,run_month,key,LY
52,2026-06-30,Amazon ARIPL,718312,0.321,1.12117,PCNO(R),0,0,0,0,...,0,0,0,0,0,NaN,2,2026-07-31,Amazon ARIPL_718312,NaN
53,2026-07-31,Amazon ARIPL,718312,0.000,0.00000,PCNO(R),1,0,0,0,...,0,0,0,0,0,NaN,3,2026-07-31,Amazon ARIPL_718312,NaN
54,2026-08-31,Amazon ARIPL,718312,0.000,0.00000,PCNO(R),1,0,0,0,...,0,0,0,0,0,0.0,3,2026-07-31,Amazon ARIPL_718312,NaN
55,2026-09-30,Amazon ARIPL,718312,0.000,0.00000,PCNO(R),1,0,0,0,...,0,0,0,0,0,0.0,3,2026-07-31,Amazon ARIPL_718312,NaN
56,2026-10-31,Amazon ARIPL,718312,0.000,0.00000,PCNO(R),1,0,0,0,...,0,0,0,0,0,0.0,4,2026-07-31,Amazon ARIPL_718312,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128630,2026-12-31,Nykaa,811021,0.000,0.00000,PADV_WIPS,1,0,0,0,...,0,0,0,0,0,0.0,4,2026-07-31,Nykaa_811021,NaN
128631,2027-01-31,Nykaa,811021,0.000,0.00000,PADV_WIPS,1,0,0,0,...,0,0,0,0,0,0.0,1,2026-07-31,Nykaa_811021,NaN
128632,2027-02-28,Nykaa,811021,0.000,0.00000,PADV_WIPS,1,0,0,0,...,0,0,0,0,0,0.0,1,2026-07-31,Nykaa_811021,NaN
128633,2027-03-31,Nykaa,811021,0.000,0.00000,PADV_WIPS,1,0,0,0,...,0,0,0,0,0,NaN,1,2026-07-31,Nykaa_811021,NaN


In [137]:
missing_df['key'].nunique()

510

In [138]:
missing_df = missing_df[['key','run_month','month_date', 'platform_name', 'parent_material_code', 'brand_code',
       'vol_in_rum']]
missing_df

,key,run_month,month_date,platform_name,parent_material_code,brand_code,vol_in_rum
52,Amazon ARIPL_718312,2026-07-31,2026-06-30,Amazon ARIPL,718312,PCNO(R),0.321
53,Amazon ARIPL_718312,2026-07-31,2026-07-31,Amazon ARIPL,718312,PCNO(R),0.000
54,Amazon ARIPL_718312,2026-07-31,2026-08-31,Amazon ARIPL,718312,PCNO(R),0.000
55,Amazon ARIPL_718312,2026-07-31,2026-09-30,Amazon ARIPL,718312,PCNO(R),0.000
56,Amazon ARIPL_718312,2026-07-31,2026-10-31,Amazon ARIPL,718312,PCNO(R),0.000
...,...,...,...,...,...,...,...
128630,Nykaa_811021,2026-07-31,2026-12-31,Nykaa,811021,PADV_WIPS,0.000
128631,Nykaa_811021,2026-07-31,2027-01-31,Nykaa,811021,PADV_WIPS,0.000
128632,Nykaa_811021,2026-07-31,2027-02-28,Nykaa,811021,PADV_WIPS,0.000
128633,Nykaa_811021,2026-07-31,2027-03-31,Nykaa,811021,PADV_WIPS,0.000


In [139]:
qtr_df = read_qtr_ind_rate_table()
qtr_df.columns = qtr_df.columns.str.lower()
qtr_df.head()


Credentials retrieved successfully for prod db.


,month_date,brand_code,qtr_ind_rate
0,2027-03-31,PA_CN_HGO,488.152
1,2027-03-31,TRU_RAWDF,800.000
2,2027-03-31,TRU_PDRFR,850.570
3,2027-03-31,TRU_OATS,177.070
4,2027-03-31,TRU_QUINO,204.750


In [140]:
len_before_merge = len(missing_df)

missing_df = missing_df.rename(columns={'material_group_code': 'brand_code'}).merge(
    qtr_df.drop('month_date', axis=1),
    on=['brand_code'],
    how='left'
)

assert len_before_merge == len(missing_df)

In [141]:
missing_df

,key,run_month,month_date,platform_name,parent_material_code,brand_code,vol_in_rum,qtr_ind_rate
0,Amazon ARIPL_718312,2026-07-31,2026-06-30,Amazon ARIPL,718312,PCNO(R),0.321,349274.001420
1,Amazon ARIPL_718312,2026-07-31,2026-07-31,Amazon ARIPL,718312,PCNO(R),0.000,349274.001420
2,Amazon ARIPL_718312,2026-07-31,2026-08-31,Amazon ARIPL,718312,PCNO(R),0.000,349274.001420
3,Amazon ARIPL_718312,2026-07-31,2026-09-30,Amazon ARIPL,718312,PCNO(R),0.000,349274.001420
4,Amazon ARIPL_718312,2026-07-31,2026-10-31,Amazon ARIPL,718312,PCNO(R),0.000,349274.001420
...,...,...,...,...,...,...,...,...
8707,Nykaa_811021,2026-07-31,2026-12-31,Nykaa,811021,PADV_WIPS,0.000,366.484998
8708,Nykaa_811021,2026-07-31,2027-01-31,Nykaa,811021,PADV_WIPS,0.000,366.484998
8709,Nykaa_811021,2026-07-31,2027-02-28,Nykaa,811021,PADV_WIPS,0.000,366.484998
8710,Nykaa_811021,2026-07-31,2027-03-31,Nykaa,811021,PADV_WIPS,0.000,366.484998


In [142]:
missing_df['month_date'] = pd.to_datetime(missing_df['month_date'])
missing_df['run_month'] = pd.to_datetime(missing_df['run_month'])


mappings = {}

for run_month in missing_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 11):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   
missing_df['M month'] = missing_df.apply(
    lambda x: mappings[x['run_month']].get(
        x['month_date']
    ), axis=1
)

brand_md_df = pd.read_excel(r"/data/aman_singh/mt_forecast/Brand_metadata.xlsx")
len_before_merge = len(missing_df)
missing_df = missing_df.merge(
    brand_md_df,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(missing_df)

# Merge 70th percentile Prophet predictions

assert missing_df.duplicated(
    subset=['run_month', 'month_date', 'key']
).sum() == 0
missing_df.drop('vol_in_rum', axis=1, inplace=True)

In [143]:
missing_df['M month'].unique()

array([None, 'M', 'M+1', 'M+2', 'M+3', 'M+4', 'M+5', 'M+6', 'M+7', 'M+8',
       'M+9'], dtype=object)

In [144]:
offtake_df['run_month'] = pd.to_datetime(offtake_df['run_month'])
offtake_df['month_date'] = pd.to_datetime(offtake_df['month_date'])
assert offtake_df.duplicated(subset=['key', 'month_date']).sum() == 0
len_before_merge = len(missing_df)
missing_df = missing_df.merge(
    offtake_df[['key', 'month_date', 'vol_in_rum']],
    on=['month_date', 'key'],
    how='left'
)
assert len(missing_df) == len_before_merge

missing_df['vol_in_rum'].fillna(0, inplace=True)
for col in [ 'vol_in_rum']:
    missing_df[col] = missing_df[col].clip(lower=0)
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)

In [145]:
missing_df['P3M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=3, min_periods=1).mean()

missing_df['P6M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=6, min_periods=3).mean()

missing_df['LY P3M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=3, min_periods=3).mean()

missing_df['LY P6M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=6, min_periods=6).mean()

In [146]:

missing_df['LY P3M_copy'] = missing_df['LY P3M'].copy()
missing_df['P3M Max'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).max()
missing_df['P3M Top 2 Mean'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sort(x)[-2:].mean(), raw=True) 

# lambda x: x.nlargest(2).mean(), raw=False
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['MoM P3M growth'] = (
    missing_df.groupby(['run_month', 'key'])['P3M']
      .pct_change() * 100
)
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['MoM P3M growth_lag_1'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(1)
missing_df['MoM P3M growth_lag_2'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(2)


missing_df['>=20%_3M_inc_month_count'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['MoM P3M growth'].shift(0)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sum(x >= 20), raw=True) 
missing_df['Avg(P3M Mean, Max)'] = missing_df[['P3M', 'P3M Max']].mean(axis=1)
for col in ['P3M', 'P6M', 'LY P3M', 'P3M Max', 'P3M Top 2 Mean', 
            'MoM P3M growth', '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)',
            'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2' ]:
    # if not 'LY' in col:  'LY P6M',
    missing_df.loc[missing_df['month_date'] > missing_df['run_month'], [col]] = np.nan
    missing_df[col] = missing_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )


# for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
#     if not 'LY' in col:
#         missing_df.loc[missing_df['month_date'] > missing_df['run_month'], [col]] = np.nan
#         missing_df[col] = missing_df.groupby(['key'], as_index = True, group_keys = False)[col].apply(
#             lambda x: x.ffill()
#         )
# missing_df.to_csv('collate_check.csv', index=False)
for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
    missing_df[f'{col}_value'] = missing_df[col] * missing_df['qtr_ind_rate'] / (10 ** 7)
missing_df['vol_in_rum_value'] = missing_df['qtr_ind_rate'] * missing_df['vol_in_rum'] / (10 ** 7)
value_cols = [col for col in missing_df.columns if 'value' in col]
value_cols
for col in value_cols:
    try:
        assert missing_df[col].min() >= 0
    except:
        print(col)
    

    # missing_df[col] = missing_df[col] / (10 ** 7)
missing_df
assert missing_df.duplicated(
    subset=['run_month', 'brand_code', 'key', 'month_date']
).sum() == 0
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['LY'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(12)

missing_df['LLY'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(24)


missing_df['LY value'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(12)

missing_df['LLY value'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(24)


missing_df['OT_Value_in_Cr_lag_1'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(1)

missing_df['OT_Value_in_Cr_lag_2'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(2)

missing_df['OT_Value_in_Cr_lag_3'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(3)
for col in ['OT_Value_in_Cr_lag_1', 'OT_Value_in_Cr_lag_2', 'OT_Value_in_Cr_lag_3']:
    # if not 'LY' in col:  'LY P6M',
    missing_df.loc[missing_df['month_date'] > missing_df['run_month'], [col]] = np.nan
    missing_df[col] = missing_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

missing_df.reset_index(drop=True, inplace=True)

# brand_class_df = pd.read_excel(r"/data/aman_singh/acuuracy_check/Brand_Classification.xlsb")
# brand_class_df.head()
# brand_class_df.columns = ['brand_code', 'class']

# len_before_merge = len(missing_df)
# missing_df = missing_df.merge(
#     brand_class_df, 
#     on=['brand_code'],
#     how='left'
# )
# assert len_before_merge == len(missing_df)
# del len_before_merge
# missing_df['class'].isna().sum()
# missing_df['class'].unique()

LY P3M_value


In [147]:
pd.set_option('display.max_columns', None)

In [148]:
# missing_df[
#     # (missing_df['channel'].isin(['MT', 'QCOM'])) & 
#     # (missing_df['month_date'] > '2024-06-30') &
#     (missing_df['M month'].notna())
#     # (missing_df['class'].isin(['B', 'C']))
# ].to_csv('missing_combinations_QCOM_Chain_city_PSKU_Offtakes.csv', index=False)

In [149]:
model_file

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3
0,Amazon RK_718472,2023-11-30,0.000000e+00,0.51,1.125,NaN,0.487286,0.000000e+00,0.000025,0.000056,NaN,0.000024,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,0.000000,4,496.828458,0.000022,0.45,0.000022,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,1.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,NaN,NaN,0.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Amazon RK_718472,2023-12-31,4.499998e-01,0.51,1.125,NaN,0.470571,2.235727e-05,0.000025,0.000056,NaN,0.000023,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,NaN,4,496.828458,0.000000,0.00,0.000000,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,1.0,0.0,0.0,2026-07-31,None,Hair Oils,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000022,NaN,NaN
2,Amazon RK_718472,2024-01-31,1.431689e-01,0.51,1.125,NaN,1.137857,7.113037e-06,0.000025,0.000056,NaN,0.000057,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,2.400000,1,496.828458,0.000054,1.08,0.000054,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,NaN,NaN,1.08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000022,NaN
3,Amazon RK_718472,2024-02-29,7.860552e-01,0.51,1.125,NaN,1.496571,3.905346e-05,0.000025,0.000056,NaN,0.000074,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,7.200000,1,496.828458,0.000080,1.62,0.000080,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,NaN,NaN,1.62,0.51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.51,0.000025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000054,0.000000,0.000022
4,Amazon RK_718472,2024-03-31,7.437929e-01,0.90,1.125,NaN,2.247429,3.695375e-05,0.000045,0.000056,NaN,0.000112,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,5.294118,1,496.828458,0.000134,2.70,0.000134,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,NaN,NaN,2.70,0.90,NaN,NaN,NaN,NaN,NaN,NaN,76.470588,NaN,NaN,NaN,0.90,0.000045,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000080,0.000054,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114546,Big Basket_719192,2026-10-31,-1.258270e-309,0.00,0.000,NaN,0.000000,-1.258270e-314,0.000000,0.000000,NaN,0.000000,719192,Big Basket,VEG_CLEAN,0.0,0.0,0.0,0.0,0.0,NaN,4,100.000000,0.000000,0.00,0.000000,2026-06-30,4.665446,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-07-31,M+3,Health & Hygiene,NaN,NaN,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,-100.000000,-100.0,-10

In [150]:
model_file['skipped'] = 0
missing_df['skipped'] = 1
final_df = pd.concat([model_file,missing_df])
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped
0,Amazon RK_718472,2023-11-30,0.000000,0.51,1.125,NaN,0.487286,0.000000,0.000025,0.000056,NaN,0.000024,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,0.000000,4.0,496.828458,0.000022,0.45,0.000022,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,1.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,NaN,NaN,0.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,Amazon RK_718472,2023-12-31,0.450000,0.51,1.125,NaN,0.470571,0.000022,0.000025,0.000056,NaN,0.000023,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,NaN,4.0,496.828458,0.000000,0.00,0.000000,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,1.0,0.0,0.0,2026-07-31,None,Hair Oils,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000022,NaN,NaN,0
2,Amazon RK_718472,2024-01-31,0.143169,0.51,1.125,NaN,1.137857,0.000007,0.000025,0.000056,NaN,0.000057,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,2.400000,1.0,496.828458,0.000054,1.08,0.000054,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,NaN,NaN,1.08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000022,NaN,0
3,Amazon RK_718472,2024-02-29,0.786055,0.51,1.125,NaN,1.496571,0.000039,0.000025,0.000056,NaN,0.000074,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,7.200000,1.0,496.828458,0.000080,1.62,0.000080,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,NaN,NaN,1.62,0.51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.51,0.000025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000054,0.000000,0.000022,0
4,Amazon RK_718472,2024-03-31,0.743793,0.90,1.125,NaN,2.247429,0.000037,0.000045,0.000056,NaN,0.000112,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,5.294118,1.0,496.828458,0.000134,2.70,0.000134,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,NaN,NaN,2.70,0.90,NaN,NaN,NaN,NaN,NaN,NaN,76.470588,NaN,NaN,NaN,0.90,0.000045,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000080,0.000054,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8707,Myntra_811169,2026-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,811169,Myntra,SW_SGPRF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1712.605337,0.000000,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-31,M+5,Male Grooming,NaN,NaN,0.00,5.22,2.088,NaN,NaN,NaN,NaN,NaN,126.562500,inf,NaN,NaN,5.22,0.000894,0.000358,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000999,0.000789,NaN,1
8708,Myntra_811169,2027-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN

In [151]:
final_df[final_df.select_dtypes(include='number').columns] = final_df.select_dtypes(include='number').fillna(0)
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped
0,Amazon RK_718472,2023-11-30,0.000000,0.51,1.125,0.0,0.487286,0.000000,0.000025,0.000056,0.0,0.000024,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.000000,4.0,496.828458,0.000022,0.45,0.000022,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,1.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,0.45,0.00,0.000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.00,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0
1,Amazon RK_718472,2023-12-31,0.450000,0.51,1.125,0.0,0.470571,0.000022,0.000025,0.000056,0.0,0.000023,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.000000,4.0,496.828458,0.000000,0.00,0.000000,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,1.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,0.00,0.00,0.000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.00,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000022,0.000000,0.000000,0
2,Amazon RK_718472,2024-01-31,0.143169,0.51,1.125,0.0,1.137857,0.000007,0.000025,0.000056,0.0,0.000057,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,2.400000,1.0,496.828458,0.000054,1.08,0.000054,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,1.08,0.00,0.000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.00,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000022,0.000000,0
3,Amazon RK_718472,2024-02-29,0.786055,0.51,1.125,0.0,1.496571,0.000039,0.000025,0.000056,0.0,0.000074,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,7.200000,1.0,496.828458,0.000080,1.62,0.000080,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,1.62,0.51,0.000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.51,0.000025,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000054,0.000000,0.000022,0
4,Amazon RK_718472,2024-03-31,0.743793,0.90,1.125,0.0,2.247429,0.000037,0.000045,0.000056,0.0,0.000112,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,5.294118,1.0,496.828458,0.000134,2.70,0.000134,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,2.70,0.90,0.000,0.0,0.0,0.0,0.0,0.0,76.470588,0.0,0.0,0.0,0.90,0.000045,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000080,0.000054,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8707,Myntra_811169,2026-12-31,0.000000,0.00,0.000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,811169,Myntra,SW_SGPRF,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,1712.605337,0.000000,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-07-31,M+5,Male Grooming,0.0,0.0,0.00,5.22,2.088,0.0,0.0,0.0,0.0,0.0

In [152]:
final_df['month_date'] = pd.to_datetime(final_df['month_date'])
final_df['run_month'] = pd.to_datetime(final_df['run_month'])
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped
0,Amazon RK_718472,2023-11-30,0.000000,0.51,1.125,0.0,0.487286,0.000000,0.000025,0.000056,0.0,0.000024,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.000000,4.0,496.828458,0.000022,0.45,0.000022,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,1.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,0.45,0.00,0.000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.00,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0
1,Amazon RK_718472,2023-12-31,0.450000,0.51,1.125,0.0,0.470571,0.000022,0.000025,0.000056,0.0,0.000023,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.000000,4.0,496.828458,0.000000,0.00,0.000000,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,1.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,0.00,0.00,0.000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.00,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000022,0.000000,0.000000,0
2,Amazon RK_718472,2024-01-31,0.143169,0.51,1.125,0.0,1.137857,0.000007,0.000025,0.000056,0.0,0.000057,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,2.400000,1.0,496.828458,0.000054,1.08,0.000054,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,1.08,0.00,0.000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.00,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000022,0.000000,0
3,Amazon RK_718472,2024-02-29,0.786055,0.51,1.125,0.0,1.496571,0.000039,0.000025,0.000056,0.0,0.000074,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,7.200000,1.0,496.828458,0.000080,1.62,0.000080,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,1.62,0.51,0.000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.51,0.000025,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000054,0.000000,0.000022,0
4,Amazon RK_718472,2024-03-31,0.743793,0.90,1.125,0.0,2.247429,0.000037,0.000045,0.000056,0.0,0.000112,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,5.294118,1.0,496.828458,0.000134,2.70,0.000134,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,2.70,0.90,0.000,0.0,0.0,0.0,0.0,0.0,76.470588,0.0,0.0,0.0,0.90,0.000045,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000080,0.000054,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8707,Myntra_811169,2026-12-31,0.000000,0.00,0.000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,811169,Myntra,SW_SGPRF,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,1712.605337,0.000000,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-07-31,M+5,Male Grooming,0.0,0.0,0.00,5.22,2.088,0.0,0.0,0.0,0.0,0.0

Heuristic new approac

In [153]:
# pip install pymannkendall

identify events

In [154]:
final_df['month_date'] = pd.to_datetime(final_df['month_date'])
final_df['run_month'] = pd.to_datetime(final_df['run_month'])

In [155]:
import numpy as np 
import pandas as pd 
EVENT_MONTHS = [9, 10, 11] # Sep, Oct, Nov
def detect_event_months(df_grp):
    df_grp = df_grp.sort_values("month_date").copy()
    run_month = df_grp["run_month"].max()

    # only historical data
    hist = df_grp[df_grp["month_date"] < run_month].copy()

    # initialize
    df_grp["event_month_flag"] = 0
    df_grp["event_uplift_factor"] = 0.0

    if len(hist) < 12:
        df_grp["event_sensitive_flag"] = 0
        return df_grp

    event_sensitive = 0

    # loop year-wise
    for year in hist["month_date"].dt.year.unique():

        year_df = hist[hist["month_date"].dt.year == year]

        for _, row in year_df.iterrows():

            month = row["month_date"].month

            if month not in EVENT_MONTHS:
                continue

            curr_date = row["month_date"]

            # previous 12 months before this month
            prev_12m = hist[
                (hist["month_date"] < curr_date) &
                (hist["month_date"] >= curr_date - pd.DateOffset(months=12)) &
                (~hist["month_date"].dt.month.isin(EVENT_MONTHS))  # 
            ]

            if len(prev_12m) < 6:
                continue

            prev_12m_avg = prev_12m["vol_in_rum_value"].mean()

            if prev_12m_avg <= 0 or np.isnan(prev_12m_avg):
                continue

            uplift = row["vol_in_rum_value"] / prev_12m_avg

            if uplift > 2:
                mask = df_grp["month_date"] == curr_date
                df_grp.loc[mask, "event_month_flag"] = 1
                df_grp.loc[mask, "event_uplift_factor"] = uplift
                event_sensitive = 1

    df_grp["event_sensitive_flag"] = event_sensitive
    return df_grp


In [156]:
final_df = (
    final_df
    .groupby(
        ["platform_name", "parent_material_code", "run_month"],
        group_keys=False
    )
    .apply(detect_event_months)
)


In [157]:
final_df[(final_df['event_month_flag'] == 1) & (final_df['platform_name'] == 'Flipkart National') & ((final_df['parent_material_code'] == 718939))]

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag
75442,Flipkart National_718939,2023-09-30,117.126136,111.709333,86.266000,171.199718,218.075562,1.376744,1.313073,1.014003,2.012345,2.563341,718939,Flipkart National,SAFF ACTV,0.0,0.0,0.0,1.0,0.0,2.617885,3.0,117543.724493,2.186454,186.012000,2.186454,2026-06-30,0.763883,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Saffola Oils,183.826424,175.667602,186.012000,111.709333,86.266000,0.000000,0.000000,0.000000,99.329333,90.655333,12.463589,21.160915,34.787470,2.0,105.519333,1.313073,1.014003,0.000000,0.000000,2.160764,2.064862,0.000,0.000,0.000000,0.000000,1.593188,0.996912,1.349120,0,1,2.251920,1
75443,Flipkart National_718939,2023-10-31,117.126136,135.454667,108.718000,344.497859,313.313809,1.376744,1.592185,1.277912,4.049356,3.682807,718939,Flipkart National,SAFF ACTV,1.0,0.0,0.0,0.0,0.0,2.498109,4.0,117543.724493,4.293919,365.304000,4.293919,2026-06-30,0.763883,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Saffola Oils,360.240750,350.944186,365.304000,135.454667,108.718000,0.000000,0.000000,0.000000,111.709333,105.519333,21.256356,12.463589,21.160915,2.0,123.582000,1.592185,1.277912,0.000000,0.000000,4.234404,4.125129,0.000,0.000,0.000000,0.000000,2.186454,1.593188,0.996912,0,1,4.422486,1
75444,Flipkart National_718939,2023-11-30,117.126136,228.952000,164.140667,172.254414,181.393600,1.376744,2.691187,1.929371,2.024743,2.132168,718939,Flipkart National,SAFF ACTV,0.0,1.0,0.0,0.0,0.0,0.661388,4.0,117543.724493,2.220166,188.880000,2.220166,2026-06-30,0.763883,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Saffola Oils,186.486476,178.370538,188.880000,228.952000,164.140667,0.000000,0.000000,0.000000,135.454667,123.582000,69.024815,21.256356,12.463589,2.0,182.203333,2.691187,1.929371,0.000000,0.000000,2.192031,2.096634,0.000,0.000,0.000000,0.000000,4.293919,2.186454,1.593188,0,1,2.286641,1
75454,Flipkart National_718939,2024-09-30,164.360187,115.513333,84.795333,328.674125,300.122914,1.931951,1.357787,0.996716,3.863358,3.527757,718939,Flipkart National,SAFF ACTV,1.0,0.0,0.0,1.0,0.0,1.872679,3.0,117543.724493,3.490061,296.916000,3.490061,2026-06-30,0.763883,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Saffola Oils,345.444927,337.877625,296.916000,115.513333,84.795333,111.709333,86.266000,111.709333,99.206667,85.028667,16.437067,40.022206,31.017309,2.0,107.360000,1.357787,0.996716,1.313073,1.014003,4.060488,3.971539,186.012,0.000,2.186454,0.000000,2.056310,1.234820,0.782230,0,1,3.377611,1
75455,Flipkart National_718939,2024-10-31,287.298145,192.302667,131.576667,257.590710,228.260728,3.377009,2.260397,1.546601,3.027817,2.683062,718939,Flipkart National,SAFF ACTV,1.0,1.0,0.0,0.0,0.0,3.270130,4.0,117543.724493,2.815877,239.560000,2.815877,2026-06-30,0.763883

In [158]:
def compute_adjusted_pm(df_grp, window, column, year_shift=0):
    df_grp = df_grp.sort_values("month_date")

    run_month = df_grp["run_month"].iloc[0]

    # define cutoff
    end_date = run_month - pd.DateOffset(years=year_shift)

    # keep only eligible history (before run month & non-event)
    hist = df_grp[
        (df_grp["month_date"] < end_date) &
        (df_grp["event_month_flag"] == 0)
    ]

    if hist.empty:
        return np.nan

    # take last `window` non-event months
    hist = hist.tail(window)

    # if len(hist) < window:
    #     return np.nan   # optional, keeps behavior strict

    return hist[column].mean()


In [159]:
adj_df = final_df.groupby(
    ["platform_name", "parent_material_code", "run_month"]
).apply(
    lambda x: pd.Series({
        "P3M_adj": compute_adjusted_pm(x, 3,'vol_in_rum',0),
        "P6M_adj": compute_adjusted_pm(x, 6,'vol_in_rum',0),
        "P3M_adj_value": compute_adjusted_pm(x, 3,'vol_in_rum_value',0),
        "P6M_adj_value": compute_adjusted_pm(x, 6,'vol_in_rum_value',0)
    })
).reset_index()

final_df = final_df.merge(
    adj_df,
    on=["platform_name", "parent_material_code", "run_month"],
    how="left"
)

In [160]:
adj_ly_df = final_df.groupby(
    ["platform_name", "parent_material_code", "run_month"]
).apply(
    lambda x: pd.Series({
        "LY_P3M_adj": compute_adjusted_pm(x, 3,'vol_in_rum',year_shift=1),
        "LY_P6M_adj": compute_adjusted_pm(x, 6,'vol_in_rum', year_shift=1),
        "LY_P3M_adj_value": compute_adjusted_pm(x, 3,'vol_in_rum_value', year_shift=1),
        "LY_P6M_adj_value": compute_adjusted_pm(x, 6,"vol_in_rum_value", year_shift=1)
    })
).reset_index()

In [161]:
final_df = final_df.merge(
    adj_ly_df,
    on=["platform_name", "parent_material_code", "run_month"],
    how="left"
)
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value
0,Amazon RK_718472,2023-11-30,0.000000e+00,0.51,1.125,0.0,0.487286,0.000000e+00,0.000025,0.000056,0.0,0.000024,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,4.0,496.828458,0.000022,0.45,0.000022,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,1.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,0.45,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.000000,0.0,0.0,0.0,0.0
1,Meesho_718472,2025-12-31,0.000000e+00,0.00,0.000,0.0,0.000000,0.000000e+00,0.000000,0.000000,0.0,0.000000,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0.002119,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,42.66,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1,0,0.0,0,0.36,2.34,0.000018,0.000116,NaN,NaN,NaN,NaN
2,Amazon RK_718472,2023-12-31,4.499998e-01,0.51,1.125,0.0,0.470571,2.235727e-05,0.000025,0.000056,0.0,0.000023,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,4.0,496.828458,0.000000,0.00,0.000000,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,1.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000022,0.000000,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.000000,0.0,0.0,0.0,0.0
3,Meesho_718472,2026-01-31,0.000000e+00,0.00,0.000,0.0,0.000000,0.000000e+00,0.000000,0.000000,0.0,0.000000,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0.000456,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,9.18,42.66,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,42.66,0.002119,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.002119,0.000000,0.0,1,0,0.0,0,0.36,2.34,0.000018,0.000116,NaN,NaN,NaN,NaN
4,Amazon RK_718472,2024-01-31,1.431689e-01,0.51,1.125,0.0,1.137857,7.113037e-06,0.000025,0.000056,0.0,0.000057,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,2.4,1.0,496.828458,0.000054,1.08,0.000054,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,1.08,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000022,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.000000,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123258,Big Basket_719192,2026-10-31,-1.258270e-309,0.00,0

In [162]:
qtr_df = read_qtr_ind_rate_table()
qtr_df.columns = qtr_df.columns.str.lower()
qtr_df.head()


len_before_merge = len(final_df)

final_df = final_df.merge(
    qtr_df.drop('month_date', axis=1),
    on=['brand_code'],
    how='left'
)

assert len_before_merge == len(final_df)


Credentials retrieved successfully for prod db.


In [163]:
import pandas as pd
import numpy as np
import pymannkendall as mk

def detect_trend_for_group(df_grp):
    """
    Detect final trend flag and p3m_slope_flag separately.
    Must contain 'month_date', 'vol_in_rum', 'run_month'
    """

    # ---------- 1. Sort ----------
    df_grp = df_grp.sort_values("month_date")

    # ---------- 2. Identify run_month ----------
    run_month = df_grp["run_month"].max()

    # actual data = months < run_month
    df_actual = df_grp[df_grp["month_date"] < run_month]

    # if no actual data → no trend
    if df_actual.empty or len(df_actual) < 4:
        return pd.Series({"trend_flag": 0, "p3m_slope_flag": 0})

    # ---------- 3. MK Trend ----------
    series = df_actual["vol_in_rum_value"].astype(float)

    try:
        mk_result = mk.original_test(series)
        if mk_result.trend == "increasing":
            mk_trend = 1
        elif mk_result.trend == "decreasing":
            mk_trend = -1
        else:
            mk_trend = 0
    except:
        mk_trend = 0

    # ---------- 4. P3M Slope ----------
    # last 4 months → take last 3 with shift
    #shifted_series = series.shift(1).dropna()

    p3m_values = series.tail(3).values
    #print(p3m_values)

    if len(p3m_values) < 3:
        slope_flag = 0
    else:
        x = np.arange(3)
        slope = np.polyfit(x, p3m_values, 1)[0]
        slope_flag = 1 if slope > 0 else (-1 if slope < 0 else 0)
        

    return pd.Series({
        "trend_flag": mk_trend,
        "p3m_slope_flag": slope_flag
    })


# ---------------------------------------------------------
# APPLY ON ENTIRE DATASET
# ---------------------------------------------------------

# trend_df = final_df.groupby(
#     ["platform_name", "parent_material_code"]
# ).apply(detect_trend_for_group).reset_index()

# trend_df = final_df[final_df['key'] == 'Zepto_721898'].groupby(
#     ["platform_name", "parent_material_code", "run_month"]
# ).apply(detect_trend_for_group).reset_index()
trend_df = final_df.groupby(
    ["platform_name", "parent_material_code", "run_month"]
).apply(detect_trend_for_group).reset_index()

In [164]:
trend_df["final_trend"] = np.where(
    (trend_df["trend_flag"] == 1) & (trend_df["p3m_slope_flag"] == 1), 1,
    np.where(
        (trend_df["trend_flag"] == -1) & (trend_df["p3m_slope_flag"] == -1), -1,
        0
    )
)
trend_df

,platform_name,parent_material_code,run_month,trend_flag,p3m_slope_flag,final_trend
0,Amazon ARIPL,718288,2026-07-31,1,1,1
1,Amazon ARIPL,718312,2026-07-31,0,0,0
2,Amazon ARIPL,718321,2026-07-31,-1,0,0
3,Amazon ARIPL,718322,2026-07-31,0,-1,0
4,Amazon ARIPL,718323,2026-07-31,-1,0,0
...,...,...,...,...,...,...
3191,Nykaa,810605,2026-07-31,-1,0,0
3192,Nykaa,810673,2026-07-31,0,-1,0
3193,Nykaa,810674,2026-07-31,0,1,0
3194,Nykaa,811019,2026-07-31,0,-1,0


## detect seasonality

In [165]:
from statsmodels.tsa.stattools import acf
import numpy as np
import pandas as pd

def detect_yearly_seasonality(df_grp, threshold=0.3):
    """
    Detects yearly seasonality using ACF at lag=12 only.
    Uses vol_in_rum as the metric.
    """
    df_grp = df_grp.sort_values("month_date")
    run_month = df_grp["run_month"].max()

    # actual data = months < run_month
    df_actual = df_grp[df_grp["month_date"] < run_month]
    series = df_actual["vol_in_rum"].astype(float).values

    # Need at least 18 points to compare last year vs this year
    if len(series) < 18:
        return 0

    # Compute ACF up to lag-12
    acf_vals = acf(series, nlags=12, fft=False)

    lag12_acf = acf_vals[12]

    # absolute ACF because seasonal correlation can be negative as well
    if abs(lag12_acf) >= threshold:
        return 1
    else:
        return 0
    

seasonality_df = final_df.groupby(
    ["platform_name", "brand_code", 'run_month']
).apply(detect_yearly_seasonality).reset_index(name="seasonality_flag")

seasonality_df



,platform_name,brand_code,run_month,seasonality_flag
0,Amazon ARIPL,CO_SO_VCN,2026-07-31,0
1,Amazon ARIPL,PA-BDYLOT,2026-07-31,0
2,Amazon ARIPL,PABABY_ML,2026-07-31,0
3,Amazon ARIPL,PADV-HRCR,2026-07-31,0
4,Amazon ARIPL,PA_EXT_ML,2026-07-31,0
...,...,...,...,...
601,Nykaa,SW HRGEL,2026-07-31,0
602,Nykaa,SW NOGAS,2026-07-31,0
603,Nykaa,SW STLDEO,2026-07-31,0
604,Nykaa,SW_HR_WAX,2026-07-31,0


In [166]:
# seasonality_df.to_csv('seasonal_ecom.csv')

In [167]:
import numpy as np
import pandas as pd

def compute_thresholds(df_grp):
    """
    df_grp MUST contain:
    - month_date
    - vol_in_rum
    - run_month

    Returns: lower_threshold, upper_threshold, mean, std
    """

    df_grp = df_grp.sort_values("month_date")
    run_month = df_grp["run_month"].max()

    # --- Use ONLY actual data (strictly before run month)
    df_actual = df_grp[df_grp["month_date"] < run_month]

    series = df_actual["vol_in_rum_value"].astype(float).values

    # If no real data → return zeros
    if len(series) == 0:
        return pd.Series({
            "lower_threshold": 0,
            "upper_threshold": 0,
            "mean_value": 0,
            "std_value": 0
        })

    # --- Take last 12 months OR all available
    if len(series) > 12:
        series = series[-12:]

    mean_val = np.mean(series)
    std_val = np.std(series)

    # --- SPECIAL CASE: ≤3 data points
    if len(series) <= 3:
        lower = 0.5 * mean_val
        upper = 2 * mean_val

        return pd.Series({
            "lower_threshold": lower,
            "upper_threshold": upper,
            "mean_value": mean_val,
            "std_value": std_val
        })

    # --- Normal case (std can be zero also)
    lower = max(0,mean_val - 2*std_val)
    upper = mean_val + 3*std_val

    return pd.Series({
        "lower_threshold": lower,
        "upper_threshold": upper,
        "mean_value": mean_val,
        "std_value": std_val
    })

threshold_df = final_df.groupby(
    ["platform_name", "parent_material_code", "run_month"]
).apply(compute_thresholds).reset_index()

threshold_df.head()


,platform_name,parent_material_code,run_month,lower_threshold,upper_threshold,mean_value,std_value
0,Amazon ARIPL,718288,2026-07-31,0.116940,0.608050,0.313384,0.098222
1,Amazon ARIPL,718312,2026-07-31,0.005606,0.022423,0.011212,0.000000
2,Amazon ARIPL,718321,2026-07-31,0.000000,0.000000,0.000000,0.000000
3,Amazon ARIPL,718322,2026-07-31,0.000000,0.425722,0.153197,0.090842
4,Amazon ARIPL,718323,2026-07-31,0.000000,0.000000,0.000000,0.000000


In [168]:
trend_df = trend_df.merge(threshold_df, on = ['platform_name', 'parent_material_code', 'run_month'], how = 'left')
trend_df

,platform_name,parent_material_code,run_month,trend_flag,p3m_slope_flag,final_trend,lower_threshold,upper_threshold,mean_value,std_value
0,Amazon ARIPL,718288,2026-07-31,1,1,1,0.116940,0.608050,0.313384,0.098222
1,Amazon ARIPL,718312,2026-07-31,0,0,0,0.005606,0.022423,0.011212,0.000000
2,Amazon ARIPL,718321,2026-07-31,-1,0,0,0.000000,0.000000,0.000000,0.000000
3,Amazon ARIPL,718322,2026-07-31,0,-1,0,0.000000,0.425722,0.153197,0.090842
4,Amazon ARIPL,718323,2026-07-31,-1,0,0,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...
3191,Nykaa,810605,2026-07-31,-1,0,0,0.000000,0.000000,0.000000,0.000000
3192,Nykaa,810673,2026-07-31,0,-1,0,0.001815,0.003242,0.002386,0.000285
3193,Nykaa,810674,2026-07-31,0,1,0,0.000093,0.001661,0.000720,0.000314
3194,Nykaa,811019,2026-07-31,0,-1,0,0.000109,0.001233,0.000558,0.000225


In [169]:
# trend_df.to_csv('t_s_t_df_ecom.csv')

In [170]:
final_df = final_df.merge(seasonality_df, on = ["platform_name", "brand_code", 'run_month'], how = 'left')
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,ratio_last_year,quarter,qtr_ind_rate_x,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value,qtr_ind_rate_y,seasonality_flag
0,Amazon RK_718472,2023-11-30,0.000000e+00,0.51,1.125,0.0,0.487286,0.000000e+00,0.000025,0.000056,0.0,0.000024,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,4.0,496.828458,0.000022,0.45,0.000022,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,1.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,0.45,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.000000,0.0,0.0,0.0,0.0,496.828458,0
1,Meesho_718472,2025-12-31,0.000000e+00,0.00,0.000,0.0,0.000000,0.000000e+00,0.000000,0.000000,0.0,0.000000,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0.002119,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,42.66,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1,0,0.0,0,0.36,2.34,0.000018,0.000116,NaN,NaN,NaN,NaN,496.828458,0
2,Amazon RK_718472,2023-12-31,4.499998e-01,0.51,1.125,0.0,0.470571,2.235727e-05,0.000025,0.000056,0.0,0.000023,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,4.0,496.828458,0.000000,0.00,0.000000,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,1.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000022,0.000000,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.000000,0.0,0.0,0.0,0.0,496.828458,0
3,Meesho_718472,2026-01-31,0.000000e+00,0.00,0.000,0.0,0.000000,0.000000e+00,0.000000,0.000000,0.0,0.000000,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0.000456,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,9.18,42.66,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,42.66,0.002119,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.002119,0.000000,0.0,1,0,0.0,0,0.36,2.34,0.000018,0.000116,NaN,NaN,NaN,NaN,496.828458,0
4,Amazon RK_718472,2024-01-31,1.431689e-01,0.51,1.125,0.0,1.137857,7.113037e-06,0.000025,0.000056,0.0,0.000057,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,2.4,1.0,496.828458,0.000054,1.08,0.000054,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,1.08,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000022,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.000000,0.0,0.0,0.0,0.0,496.828458,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,..

In [171]:
trend_df.columns

Index(['platform_name', 'parent_material_code', 'run_month', 'trend_flag',
       'p3m_slope_flag', 'final_trend', 'lower_threshold', 'upper_threshold',
       'mean_value', 'std_value'],
      dtype='object')

In [172]:
final_df = final_df.merge(trend_df[['platform_name', 'parent_material_code', 'run_month',
                                    'final_trend','lower_threshold', 'upper_threshold']], on = ["platform_name", "parent_material_code", 'run_month'], how = 'left')
final_df


,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,ratio_last_year,quarter,qtr_ind_rate_x,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value,qtr_ind_rate_y,seasonality_flag,final_trend,lower_threshold,upper_threshold
0,Amazon RK_718472,2023-11-30,0.000000e+00,0.51,1.125,0.0,0.487286,0.000000e+00,0.000025,0.000056,0.0,0.000024,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,4.0,496.828458,0.000022,0.45,0.000022,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,1.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,0.45,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.000000,0.0,0.0,0.0,0.0,496.828458,0,0,0.0,0.000000
1,Meesho_718472,2025-12-31,0.000000e+00,0.00,0.000,0.0,0.000000,0.000000e+00,0.000000,0.000000,0.0,0.000000,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0.002119,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,42.66,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1,0,0.0,0,0.36,2.34,0.000018,0.000116,NaN,NaN,NaN,NaN,496.828458,0,-1,0.0,0.002551
2,Amazon RK_718472,2023-12-31,4.499998e-01,0.51,1.125,0.0,0.470571,2.235727e-05,0.000025,0.000056,0.0,0.000023,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,4.0,496.828458,0.000000,0.00,0.000000,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,1.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000022,0.000000,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.000000,0.0,0.0,0.0,0.0,496.828458,0,0,0.0,0.000000
3,Meesho_718472,2026-01-31,0.000000e+00,0.00,0.000,0.0,0.000000,0.000000e+00,0.000000,0.000000,0.0,0.000000,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0.000456,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,9.18,42.66,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,42.66,0.002119,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.002119,0.000000,0.0,1,0,0.0,0,0.36,2.34,0.000018,0.000116,NaN,NaN,NaN,NaN,496.828458,0,-1,0.0,0.002551
4,Amazon RK_718472,2024-01-31,1.431689e-01,0.51,1.125,0.0,1.137857,7.113037e-06,0.000025,0.000056,0.0,0.000057,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,2.4,1.0,496.828458,0.000054,1.08,0.000054,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,1.08,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000022,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.000000,0.0,0.0,0.0,0.0,496.828458,0,0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,.

In [173]:
final_df[final_df.select_dtypes(include='number').columns] = final_df.select_dtypes(include='number').fillna(0)
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,ratio_last_year,quarter,qtr_ind_rate_x,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value,qtr_ind_rate_y,seasonality_flag,final_trend,lower_threshold,upper_threshold
0,Amazon RK_718472,2023-11-30,0.000000e+00,0.51,1.125,0.0,0.487286,0.000000e+00,0.000025,0.000056,0.0,0.000024,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,4.0,496.828458,0.000022,0.45,0.000022,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,1.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,0.45,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.000000,0.0,0.0,0.0,0.0,496.828458,0,0,0.0,0.000000
1,Meesho_718472,2025-12-31,0.000000e+00,0.00,0.000,0.0,0.000000,0.000000e+00,0.000000,0.000000,0.0,0.000000,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0.002119,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,42.66,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1,0,0.0,0,0.36,2.34,0.000018,0.000116,0.0,0.0,0.0,0.0,496.828458,0,-1,0.0,0.002551
2,Amazon RK_718472,2023-12-31,4.499998e-01,0.51,1.125,0.0,0.470571,2.235727e-05,0.000025,0.000056,0.0,0.000023,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,4.0,496.828458,0.000000,0.00,0.000000,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,1.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000022,0.000000,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.000000,0.0,0.0,0.0,0.0,496.828458,0,0,0.0,0.000000
3,Meesho_718472,2026-01-31,0.000000e+00,0.00,0.000,0.0,0.000000,0.000000e+00,0.000000,0.000000,0.0,0.000000,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0.000456,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,9.18,42.66,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,42.66,0.002119,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.002119,0.000000,0.0,1,0,0.0,0,0.36,2.34,0.000018,0.000116,0.0,0.0,0.0,0.0,496.828458,0,-1,0.0,0.002551
4,Amazon RK_718472,2024-01-31,1.431689e-01,0.51,1.125,0.0,1.137857,7.113037e-06,0.000025,0.000056,0.0,0.000057,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,2.4,1.0,496.828458,0.000054,1.08,0.000054,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,0.0,0.0,1.08,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000022,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.000000,0.0,0.0,0.0,0.0,496.828458,0,0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,.

In [174]:
final_df.drop(columns = ['big_billion_days', 'big_billion_days_lag_1', 'big_billion_days_lag_2',
       'big_billion_days_lead_1', 'big_billion_days_lead_2', 'great_indian_festival',
       'great_indian_festival_lag_1', 'great_indian_festival_lag_2',
       'great_indian_festival_lead_1', 'great_indian_festival_lead_2'], inplace = True)

In [175]:
final_df.columns

Index(['key', 'month_date', 'pred_SARIMA', 'pred_p3m', 'pred_p6m',
       'pred_prophet', 'pred_rf', 'pred_value_SARIMA', 'pred_value_p3m',
       'pred_value_p6m', 'pred_value_prophet', 'pred_value_rf',
       'parent_material_code', 'platform_name', 'brand_code',
       'ratio_last_year', 'quarter', 'qtr_ind_rate_x', 'vol_in_rum_value',
       'vol_in_rum_treated', 'vol_in_rum_value_treated', 'train_till', 'cov',
       'run', 'step', 'file_path', 'run_month', 'M month', 'portfolio',
       'pred_prophet_70%ile', 'pred_prophet_60%ile', 'vol_in_rum', 'P3M',
       'P6M', 'LY P3M', 'LY P6M', 'LY P3M_copy', 'P3M Max', 'P3M Top 2 Mean',
       'MoM P3M growth', 'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2',
       '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)', 'P3M_value',
       'P6M_value', 'LY P3M_value', 'LY P6M_value',
       'pred_prophet_70%ile_value', 'pred_prophet_60%ile_value', 'LY', 'LLY',
       'LY value', 'LLY value', 'OT_Value_in_Cr_lag_1', 'OT_Value_in_Cr_lag_2',
    

In [176]:
missing_df['LY P3M'].sum()

0.0

In [177]:
final_df = final_df.sort_values(['key', 'month_date'])

# base LY


# LY lags
final_df['ly_lag1_value'] = (
    final_df
    .groupby(['key'])['vol_in_rum_value']
    .shift(13)
)

final_df['ly_lag2_value'] = (
    final_df
    .groupby(['key'])['vol_in_rum_value']
    .shift(14)
)

# LY leads
final_df['ly_lead1_value'] = (
    final_df
    .groupby(['key'])['vol_in_rum_value']
    .shift(11)
)

final_df['ly_lead2_value'] = (
    final_df
    .groupby(['key'])['vol_in_rum_value']
    .shift(10)
)


In [178]:
final_df[final_df['month_date'] == '2026-04-30']['P3M_value'].sum()

41.345693110416995

In [179]:
brand_seas = pd.read_excel('/data/aman_singh/acuuracy_check/seasonality.xlsx', sheet_name = 'brand')
brand_seas.columns = brand_seas.columns.str.lower()
brand_seas.rename(columns={'brand':'brand_code', 'months_num':'month', 'flag':'is_seasonal_month'}, inplace=True)

final_df['month_date'] = pd.to_datetime(final_df['month_date'])
final_df['month'] = final_df['month_date'].dt.month
final_df = final_df.merge(brand_seas, on = ['brand_code', 'month'], how = 'left')
final_df['is_seasonal_month'].fillna(0, inplace=True)

psku_seas = pd.read_excel('/data/aman_singh/acuuracy_check/seasonality.xlsx', sheet_name = 'psku')
psku_seas.columns = psku_seas.columns.str.lower()
psku_seas.rename(columns={'months_num':'month', 'flag':'is_seasonal_month_psku'}, inplace=True)

final_df = final_df.merge(psku_seas[['parent_material_code', 'month','is_seasonal_month_psku']], on = ['parent_material_code', 'month'], how = 'left')
final_df['is_seasonal_month_psku'].fillna(0, inplace=True)
final_df['final_seasonal_month'] = np.where(
    (final_df['is_seasonal_month'] == 1) | (final_df['is_seasonal_month_psku'] == 1), 1, 0
)



In [180]:
final_df['run_month'] = pd.to_datetime(final_df['run_month'])
def compute_adjusted_pm(df_grp, window, column, year_shift=0):
    df_grp = df_grp.sort_values("month_date")

    run_month = df_grp["run_month"].iloc[0]

    # define cutoff
    end_date = run_month - pd.DateOffset(years=year_shift)

    # keep only eligible history (before run month & non-event)
    hist = df_grp[
        (df_grp["month_date"] < end_date) &
        (df_grp["final_seasonal_month"] == 0)
    ]

    if hist.empty:
        return np.nan

    # take last `window` non-event months
    hist = hist.tail(window)

    # if len(hist) < window:
    #     return np.nan   # optional, keeps behavior strict

    return hist[column].mean()

adj_df = final_df.groupby(
    ['key', "run_month"]
).apply(
    lambda x: pd.Series({
        "P3M_non_seasonal": compute_adjusted_pm(x, 3,'vol_in_rum',0),
        "P6M_non_seasonal": compute_adjusted_pm(x, 6,'vol_in_rum',0),
        "P3M_non_seasonal_value": compute_adjusted_pm(x, 3,'vol_in_rum_value',0),
        "P6M_non_seasonal_value": compute_adjusted_pm(x, 6,'vol_in_rum_value',0)
    })
).reset_index()
adj_df


adj_ly_df = final_df.groupby(
    ['key', "run_month"]
).apply(
    lambda x: pd.Series({
        "LY_P3M_non_seasonal": compute_adjusted_pm(x, 3,'vol_in_rum',year_shift=1),
        "LY_P6M_non_seasonal": compute_adjusted_pm(x, 6,'vol_in_rum', year_shift=1),
        "LY_P3M_non_seasonal_value": compute_adjusted_pm(x, 3,'vol_in_rum_value', year_shift=1),
        "LY_P6M_non_seasonal_value": compute_adjusted_pm(x, 6,"vol_in_rum_value", year_shift=1)
    })
).reset_index()

adj_df = adj_df.merge(adj_ly_df, on = ['key', 'run_month'], how = 'left')
#adj_df[adj_df['key'] == 'reliance_b2c_2_haryana_718488']
adj_df

,key,run_month,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value
0,Amazon ARIPL_718288,2026-07-31,23.908000,25.586000,0.331999,0.355301,14.082000,13.536000,0.195550,0.187968
1,Amazon ARIPL_718312,2026-07-31,0.321000,0.321000,0.011212,0.011212,NaN,NaN,NaN,NaN
2,Amazon ARIPL_718321,2026-07-31,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,Amazon ARIPL_718322,2026-07-31,13.070000,13.295833,0.220658,0.224470,4.088333,4.474167,0.069022,0.075536
4,Amazon ARIPL_718323,2026-07-31,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...
3191,Nykaa_810605,2026-07-31,0.000000,0.000000,0.000000,0.000000,0.000000,0.600000,0.000000,0.000074
3192,Nykaa_810673,2026-07-31,1.745333,1.738333,0.002245,0.002236,0.042000,0.042000,0.000054,0.000054
3193,Nykaa_810674,2026-07-31,0.644000,0.543667,0.000828,0.000699,NaN,NaN,NaN,NaN
3194,Nykaa_811019,2026-07-31,16.353000,15.235500,0.000599,0.000558,NaN,NaN,NaN,NaN


In [181]:
final_df.shape

(123263, 83)

In [182]:
#adj_df.to_csv('seasonal_p3m_qcom.csv', index=False)
#all[all['month_date'].isin(['2025-11-30','2025-12-31','2026-01-31'])].groupby(['key','run_month','month_date','final_seasonal_month'])['vol_in_rum'].sum().reset_index().to_csv('seasonal_month_check.csv', index=False)
final_df = final_df.merge(
    adj_df,
    on=['key'],
    how="left"
)
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,ratio_last_year,quarter,qtr_ind_rate_x,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month_x,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value,qtr_ind_rate_y,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value,month,is_seasonal_month,seasonal_months,is_seasonal_month_psku,final_seasonal_month,run_month_y,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value
0,Amazon ARIPL_718288,2023-01-31,0.436508,9.020000,9.154167,9.121589,9.505544,0.006062,0.125256,0.12712,0.126667,0.131999,718288,Amazon ARIPL,SAFF GOLD,0.994832,1.0,138865.260689,0.133866,9.640,0.133866,2026-06-30,0.484202,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-07-31,None,Saffola Oils,10.583375,9.840139,9.640,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.146966,0.136645,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0,0,0.0,1,23.908,25.586,0.331999,0.355301,14.082,13.536,0.19555,0.187968,138865.260689,0,1,0.11694,0.608050,NaN,NaN,NaN,NaN,1,0.0,NaN,0.0,0,2026-07-31,23.908,25.586,0.331999,0.355301,14.082,13.536,0.19555,0.187968
1,Amazon ARIPL_718288,2023-02-28,10.076225,9.020000,9.154167,7.666418,8.629083,0.139924,0.125256,0.12712,0.106460,0.119828,718288,Amazon ARIPL,SAFF GOLD,0.861631,1.0,138865.260689,0.109912,7.915,0.109912,2026-06-30,0.484202,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-07-31,None,Saffola Oils,9.131529,8.484011,7.915,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.126805,0.117813,0.0,0.0,0.0,0.0,0.133866,0.000000,0.000000,0,0,0.0,1,23.908,25.586,0.331999,0.355301,14.082,13.536,0.19555,0.187968,138865.260689,0,1,0.11694,0.608050,NaN,NaN,NaN,NaN,2,0.0,NaN,0.0,0,2026-07-31,23.908,25.586,0.331999,0.355301,14.082,13.536,0.19555,0.187968
2,Amazon ARIPL_718288,2023-03-31,9.843150,9.020000,9.154167,11.939881,9.587727,0.136687,0.125256,0.12712,0.165803,0.133140,718288,Amazon ARIPL,SAFF GOLD,1.177605,1.0,138865.260689,0.131991,9.505,0.131991,2026-06-30,0.484202,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-07-31,None,Saffola Oils,13.541509,12.732233,9.505,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.188045,0.176806,0.0,0.0,0.0,0.0,0.109912,0.133866,0.000000,0,0,0.0,1,23.908,25.586,0.331999,0.355301,14.082,13.536,0.19555,0.187968,138865.260689,0,1,0.11694,0.608050,NaN,NaN,NaN,NaN,3,0.0,NaN,0.0,0,2026-07-31,23.908,25.586,0.331999,0.355301,14.082,13.536,0.19555,0.187968
3,Amazon ARIPL_718288,2023-04-30,9.904503,9.020000,9.154167,5.428280,8.784717,0.137539,0.125256,0.12712,0.075380,0.121989,718288,Amazon ARIPL,SAFF GOLD,0.834181,2.0,138865.260689,0.129006,9.290,0.129006,2026-06-30,0.484202,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-07-31,None,Saffola Oils,6.861764,6.069386,9.290,9.020000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,9.020000,0.125256,0.0,0.0,0.000000,0.095286,0.084283,0.0,0.0,0.0,0.0,0.131991,0.109912,0.133866,0,0,0.0,1

In [186]:
final_df.rename(columns = {"run_month_x":'run_month'}, inplace = True)

In [187]:
all_brand = final_df.groupby(['brand_code', 'run_month','month_date'])['vol_in_rum_value'].sum().reset_index()
all_brand

,brand_code,run_month,month_date,vol_in_rum_value
0,ADV-AHO-R,2026-07-31,2023-01-31,0.383727
1,ADV-AHO-R,2026-07-31,2023-02-28,0.296534
2,ADV-AHO-R,2026-07-31,2023-03-31,0.283240
3,ADV-AHO-R,2026-07-31,2023-04-30,0.269135
4,ADV-AHO-R,2026-07-31,2023-05-31,0.242577
...,...,...,...,...
5546,VEG_CLEAN,2026-07-31,2026-10-31,0.000000
5547,VEG_CLEAN,2026-07-31,2026-11-30,0.000000
5548,VEG_CLEAN,2026-07-31,2026-12-31,0.000000
5549,VEG_CLEAN,2026-07-31,2027-01-31,0.000000


In [188]:

def detect_month_anomaly(df, brand_code, month_num, mon=8,threshold=0.25, months_window=3):
    """
    Detect if a specific month's vol_in_rum_value is >25% different 
    from past 3 months & next 3 months, and if pattern repeats in last 2 years.
    
    Parameters:
    - df: input dataframe with 'month_date', 'vol_in_rum_value'
    - brand_code: filter by this brand code
    - month_num: month to check (6, 7, 8, 9)
    - threshold: 25% difference threshold
    - months_window: number of months before and after to compare
    """
    
    df_brand = df[df['brand_code'] == brand_code].sort_values('month_date').copy()
    
    if df_brand.empty:
        return None
    
    df_brand['year'] = df_brand['month_date'].dt.year
    df_brand['month'] = df_brand['month_date'].dt.month
    
    years = sorted(df_brand['year'].unique())
    current_year = years[-1]
    past_years = [y for y in years if y < current_year][-2:]
    
    anomalies = []
    
    for year in past_years:
        df_year = df_brand[df_brand['year'] == year].sort_values('month_date')
        
        month_data = df_year[df_year['month'] == month_num]
        if month_data.empty:
            continue
        
        month_value = month_data['vol_in_rum_value'].iloc[0]
        
        past_months = [(mon - i - 1) % 12 for i in range(1, months_window + 1)]
        print(past_months)
        next_months = [(mon + i - 1) % 12 + 1 for i in range(1, months_window + 1)]
        
        past_m = df_year[df_year['month'].isin(past_months)]['vol_in_rum_value']
        next_m = df_year[df_year['month'].isin(next_months)]['vol_in_rum_value']
        
        comparison_values = pd.concat([past_m])
        
        if comparison_values.empty:
            continue
        
        pct_diffs = []
        for comp_value in comparison_values:
            if comp_value != 0:
                pct_diff = (month_value - comp_value) / comp_value
                pct_diffs.append(pct_diff)
        
        if pct_diffs:
            positive_diffs = [p for p in pct_diffs if p > 0]
            negative_diffs = [p for p in pct_diffs if p < 0]
            same_sign = len(positive_diffs) == len(pct_diffs) or len(negative_diffs) == len(pct_diffs)
            is_anomaly = len([p for p in pct_diffs if abs(p) > threshold]) == len(pct_diffs) and same_sign
        else:
            is_anomaly = False

        anomalies.append({
            'brand_code': brand_code,
            'month': month_num,
            'year': year,
            'month_value': month_value,
            'num_months_compared': len(comparison_values),
            'pct_diffs_from_each': pct_diffs,
            'min_pct_diff': min(pct_diffs) * 100 if pct_diffs else None,
            'max_pct_diff': max(pct_diffs) * 100 if pct_diffs else None,
            'is_anomaly': is_anomaly,
            'direction': 'higher' if month_value > comparison_values.mean() else 'lower'
        })
    
    if len(anomalies) == 2:
        pattern_repeats = anomalies[0]['is_anomaly'] and anomalies[1]['is_anomaly']
        return pd.DataFrame(anomalies), pattern_repeats
    
    return pd.DataFrame(anomalies), False


# Check months 6, 7, 8, 9
brands = all_brand['brand_code'].unique()
results = []

for month in [8, 9,10,11]:
    for brand in brands:
        df_result, repeats = detect_month_anomaly(all_brand, brand, month)
        if df_result is not None and not df_result.empty:
            df_result['pattern_repeats'] = repeats
            results.append(df_result)

anomaly_summary = pd.concat(results, ignore_index=True)
# print(anomaly_summary[anomaly_summary['pattern_repeats'] == True])

[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]
[6, 5, 4]


In [189]:
final_brands = anomaly_summary[anomaly_summary['pattern_repeats'] == True].drop_duplicates(subset=['brand_code'])[['brand_code', 'direction', 'min_pct_diff', 'max_pct_diff']]
final_brands['month_different'] = 1
final_df = final_df.merge(final_brands[['brand_code', 'month_different']], on = 'brand_code', how = 'left')
final_df['month_different'].fillna(0, inplace=True)
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,ratio_last_year,quarter,qtr_ind_rate_x,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value,qtr_ind_rate_y,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value,month,is_seasonal_month,seasonal_months,is_seasonal_month_psku,final_seasonal_month,run_month_y,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value,month_different
0,Amazon ARIPL_718288,2023-01-31,0.436508,9.020000,9.154167,9.121589,9.505544,0.006062,0.125256,0.12712,0.126667,0.131999,718288,Amazon ARIPL,SAFF GOLD,0.994832,1.0,138865.260689,0.133866,9.640,0.133866,2026-06-30,0.484202,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-07-31,None,Saffola Oils,10.583375,9.840139,9.640,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.146966,0.136645,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0,0,0.0,1,23.908,25.586,0.331999,0.355301,14.082,13.536,0.19555,0.187968,138865.260689,0,1,0.11694,0.608050,NaN,NaN,NaN,NaN,1,0.0,NaN,0.0,0,2026-07-31,23.908,25.586,0.331999,0.355301,14.082,13.536,0.19555,0.187968,1.0
1,Amazon ARIPL_718288,2023-02-28,10.076225,9.020000,9.154167,7.666418,8.629083,0.139924,0.125256,0.12712,0.106460,0.119828,718288,Amazon ARIPL,SAFF GOLD,0.861631,1.0,138865.260689,0.109912,7.915,0.109912,2026-06-30,0.484202,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-07-31,None,Saffola Oils,9.131529,8.484011,7.915,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.126805,0.117813,0.0,0.0,0.0,0.0,0.133866,0.000000,0.000000,0,0,0.0,1,23.908,25.586,0.331999,0.355301,14.082,13.536,0.19555,0.187968,138865.260689,0,1,0.11694,0.608050,NaN,NaN,NaN,NaN,2,0.0,NaN,0.0,0,2026-07-31,23.908,25.586,0.331999,0.355301,14.082,13.536,0.19555,0.187968,1.0
2,Amazon ARIPL_718288,2023-03-31,9.843150,9.020000,9.154167,11.939881,9.587727,0.136687,0.125256,0.12712,0.165803,0.133140,718288,Amazon ARIPL,SAFF GOLD,1.177605,1.0,138865.260689,0.131991,9.505,0.131991,2026-06-30,0.484202,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-07-31,None,Saffola Oils,13.541509,12.732233,9.505,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.188045,0.176806,0.0,0.0,0.0,0.0,0.109912,0.133866,0.000000,0,0,0.0,1,23.908,25.586,0.331999,0.355301,14.082,13.536,0.19555,0.187968,138865.260689,0,1,0.11694,0.608050,NaN,NaN,NaN,NaN,3,0.0,NaN,0.0,0,2026-07-31,23.908,25.586,0.331999,0.355301,14.082,13.536,0.19555,0.187968,1.0
3,Amazon ARIPL_718288,2023-04-30,9.904503,9.020000,9.154167,5.428280,8.784717,0.137539,0.125256,0.12712,0.075380,0.121989,718288,Amazon ARIPL,SAFF GOLD,0.834181,2.0,138865.260689,0.129006,9.290,0.129006,2026-06-30,0.484202,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-07-31,None,Saffola Oils,6.861764,6.069386,9.290,9.020000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,9.020000,0.125256,0.0,0.0,0.000000,0.095286,0.084283,0.0,0.0,0.0,0.0,0.131991,0

In [202]:
final_df[(final_df['M month'].notna())].to_csv('/data/aman_singh/acuuracy_check/all_combination_ecom_jul_pred.csv')

In [203]:
final_df.to_csv('/data/aman_singh/acuuracy_check/all_combination_ecom_trend.csv')

In [192]:
final_df[final_df['month_date'] == '2026-08-31']['P3M_value'].sum()

37.8391834551895

### some checks

In [193]:
query = f"""select * from {input_table}
where run_month = '2026-07-31' """

data = pd.read_sql(con=dev_conn, sql=query)
data.columns = data.columns.str.lower()
data

,month_date,platform_name,parent_material_code,vol_in_rum,indexbpm,brand_code,imputed,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,run_month
0,2023-01-31,Amazon ARIPL,718288,9.640,13.27071,SAFF GOLD,0,0,0,0,0,0,0,0,0,0,0,0.994832,1,2026-07-31
1,2023-02-28,Amazon ARIPL,718288,7.915,10.89602,SAFF GOLD,0,0,0,0,0,0,0,0,0,0,0,0.861631,1,2026-07-31
2,2023-03-31,Amazon ARIPL,718288,9.505,13.08486,SAFF GOLD,0,0,0,0,0,0,0,0,0,0,0,1.177605,1,2026-07-31
3,2023-04-30,Amazon ARIPL,718288,9.290,12.78889,SAFF GOLD,0,0,0,0,0,0,0,0,0,0,0,0.834181,2,2026-07-31
4,2023-05-31,Amazon ARIPL,718288,8.050,11.08187,SAFF GOLD,0,0,0,0,0,0,0,0,0,0,0,0.957359,2,2026-07-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128630,2026-12-31,Nykaa,811021,0.000,0.00000,PADV_WIPS,1,0,0,0,0,0,0,0,0,0,0,0.000000,4,2026-07-31
128631,2027-01-31,Nykaa,811021,0.000,0.00000,PADV_WIPS,1,0,0,0,0,0,0,0,0,0,0,0.000000,1,2026-07-31
128632,2027-02-28,Nykaa,811021,0.000,0.00000,PADV_WIPS,1,0,0,0,0,0,0,0,0,0,0,0.000000,1,2026-07-31
128633,2027-03-31,Nykaa,811021,0.000,0.00000,PADV_WIPS,1,0,0,0,0,0,0,0,0,0,0,NaN,1,2026-07-31


In [194]:
data['key'] = (
    data['platform_name'].astype(str) + '_' +
    data['parent_material_code'].astype(str)
)
data = data[data['key'].isin(trend_file_df['key'].unique())]

In [195]:
trend_file_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3
0,Amazon RK_718472,2023-11-30,0.000000e+00,0.51,1.125,NaN,0.487286,0.000000e+00,0.000025,0.000056,NaN,0.000024,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,0.000000,4,496.828458,0.000022,0.45,0.000022,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,1.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,NaN,NaN,0.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Amazon RK_718472,2023-12-31,4.499998e-01,0.51,1.125,NaN,0.470571,2.235727e-05,0.000025,0.000056,NaN,0.000023,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,NaN,4,496.828458,0.000000,0.00,0.000000,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,1.0,0.0,0.0,2026-07-31,None,Hair Oils,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000022,NaN,NaN
2,Amazon RK_718472,2024-01-31,1.431689e-01,0.51,1.125,NaN,1.137857,7.113037e-06,0.000025,0.000056,NaN,0.000057,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,2.400000,1,496.828458,0.000054,1.08,0.000054,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,NaN,NaN,1.08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000022,NaN
3,Amazon RK_718472,2024-02-29,7.860552e-01,0.51,1.125,NaN,1.496571,3.905346e-05,0.000025,0.000056,NaN,0.000074,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,7.200000,1,496.828458,0.000080,1.62,0.000080,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,NaN,NaN,1.62,0.51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.51,0.000025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000054,0.000000,0.000022
4,Amazon RK_718472,2024-03-31,7.437929e-01,0.90,1.125,NaN,2.247429,3.695375e-05,0.000045,0.000056,NaN,0.000112,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,5.294118,1,496.828458,0.000134,2.70,0.000134,2026-06-30,2.612862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-07-31,None,Hair Oils,NaN,NaN,2.70,0.90,NaN,NaN,NaN,NaN,NaN,NaN,76.470588,NaN,NaN,NaN,0.90,0.000045,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000080,0.000054,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114546,Big Basket_719192,2026-10-31,-1.258270e-309,0.00,0.000,NaN,0.000000,-1.258270e-314,0.000000,0.000000,NaN,0.000000,719192,Big Basket,VEG_CLEAN,0.0,0.0,0.0,0.0,0.0,NaN,4,100.000000,0.000000,0.00,0.000000,2026-06-30,4.665446,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-07-31,M+3,Health & Hygiene,NaN,NaN,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,-100.000000,-100.0,-10

In [196]:
import pandas as pd

as_of_date = pd.to_datetime("2026-07-31")  # month-end for Feb 2026
data['month_date'] = pd.to_datetime(data['month_date'])
filtered = data[
    
    (data['month_date'] < as_of_date) &
    (data['month_date'] >= as_of_date - pd.DateOffset(months=3))
]


In [197]:
# assert p3m equals
x = filtered.groupby(['month_date'])['vol_in_rum'].sum().reset_index()['vol_in_rum'].mean()
y = trend_file_df[trend_file_df['month_date'] == '2026-07-31']['P3M'].sum()
assert(int(x)==int(y))

In [198]:
(x,y)

(264541.47547339083, 264541.47547339054)

In [199]:
ly_end = as_of_date - pd.DateOffset(years=1)
ly_start = ly_end - pd.DateOffset(months=3)

filtered = data[
    
    (data['month_date'] < ly_end) &
    (data['month_date'] >= ly_start )
]


In [200]:
# p3m ly check may not equal but should be close
x = filtered.groupby(['month_date'])['vol_in_rum'].sum().reset_index()['vol_in_rum'].mean()
y = trend_file_df[trend_file_df['month_date'] == '2026-07-31']['LY P3M'].sum()
(x,y)

(282191.8702257667, 280707.4551994329)

In [201]:
# p3m consistency check
as_of_date = pd.to_datetime('2026-07-31')

next_3_months = pd.date_range(
    start=as_of_date + pd.offsets.MonthEnd(1),
    periods=3,
    freq='M'
)
for dt in next_3_months:
    p3m_sum = trend_file_df.loc[
        trend_file_df['month_date'] == dt, 'P3M'
    ].sum()
    
    print(f"P3M sum for {dt.date()}: {p3m_sum}")


P3M sum for 2026-08-31: 264541.47547339054
P3M sum for 2026-09-30: 264541.47547339054
P3M sum for 2026-10-31: 264541.47547339054


### The end

In [106]:
import numpy as np
import pandas as pd

df = final_df.copy()

# --------------------------------------------------
# 1. SAFE FACTOR FUNCTIONS (NO ERRORS)
# --------------------------------------------------

def safe_div(a, b):
    """Safe division: if error or b<=0 → return 1."""
    try:
        if b is None or b == 0:
            return 1
        return a / b
    except:
        return 1

# recency factor = p3m/p6m (cap at 2)
df["recency_factor"] = df.apply(
    lambda r: min(2, safe_div(r["P3M_value"], r["P6M_value"])),
    axis=1
)

def safe_shrink(r,column_name):
    try:
        ratio = r[column_name] / r["P3M_value"]
        return 1 / np.sqrt(ratio)
    except:
        return 1

df["shrink_ratio_prophet"] = df.apply(lambda r: safe_shrink(r, "pred_value_prophet"), axis=1)
df["shrink_ratio_rf"] = df.apply(lambda r: safe_shrink(r, "pred_value_rf"), axis=1)

# seasonality factor = p3m / p3mLY (cap at 2)
df["seasonality_factor"] = df.apply(
    lambda r: min(2, safe_div(r["P3M_value"], r["LY P3M_value"])),
    axis=1
)


# shrink_ratio = 1 / sqrt(max(forecast/p3m,1)) → safe



# --------------------------------------------------
# 2. HEURISTICS
# --------------------------------------------------

# Recency heuristic: max(recency_factor * P3M, forecast)
df["recency_heuristic_prophet_value"] = df.apply(
    lambda r: max(r["recency_factor"] * r["P3M_value"], r["pred_value_prophet"]),
    axis=1
)

# Seasonality heuristic: max(seasonality_value, forecast)
df["seasonality_heuristic_prophet_value"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY value'], r["pred_value_prophet"]),
    axis=1
)

# Recency + Seasonality combined
df["recency_seasonality_heuristic_prophet_value"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY value'] * r["recency_factor"],
                  r["pred_value_prophet"]),
    axis=1
)

# Non-seasonal heuristic
df["non_seasonal_heuristic_prophet_value"] = df["pred_value_prophet"] * df["shrink_ratio_prophet"]


# --------------------------------------------------
# 3. FINAL DECISION TREE + skipped condition
# --------------------------------------------------

def apply_final_logic(r):

    # Rule 1: skipped → force P3M
    if r.get("skipped", 0) == 1:
        return r["P3M_value"]

    
    # Rule 3: Heuristic combinations
    if r["final_trend"] == 1 and r["seasonality_flag"] == 0:
        return r["recency_heuristic_prophet_value"]


    if r["final_trend"] != 1 and r["seasonality_flag"] == 1:
        return r["seasonality_heuristic_prophet_value"]
    
    if r["final_trend"] == 1 and r["seasonality_flag"] == 1:
        return r["recency_seasonality_heuristic_prophet_value"]

    # Default: (0,0)
    return r["non_seasonal_heuristic_prophet_value"]


df["final_heuristic_prophet_value"] = df.apply(apply_final_logic, axis=1)


In [107]:
df["recency_heuristic_rf_value"] = df.apply(
    lambda r: max(r["recency_factor"] * r["P3M_value"], r["pred_value_rf"]),
    axis=1
)

# Seasonality heuristic: max(seasonality_value, forecast)
df["seasonality_heuristic_rf_value"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY value'], r["pred_value_rf"]),
    axis=1
)

# Recency + Seasonality combined
df["recency_seasonality_heuristic_rf_value"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY value'] * r["recency_factor"],
                  r["pred_value_rf"]),
    axis=1
)

# Non-seasonal heuristic
df["non_seasonal_heuristic_rf_value"] = df["pred_value_rf"] * df["shrink_ratio_rf"]


# --------------------------------------------------
# 3. FINAL DECISION TREE + skipped condition
# --------------------------------------------------

def apply_final_logic(r):

    # Rule 1: skipped → force P3M
    if r.get("skipped", 0) == 1:
        return r["P3M_value"]

    
    # Rule 3: Heuristic combinations
    if r["final_trend"] == 1 and r["seasonality_flag"] == 0:
        return r["recency_heuristic_rf_value"]


    if r["final_trend"] != 1 and r["seasonality_flag"] == 1:
        return r["seasonality_heuristic_rf_value"]
    
    if r["final_trend"] == 1 and r["seasonality_flag"] == 1:
        return r["recency_seasonality_heuristic_rf_value"]

    # Default: (0,0)
    return r["non_seasonal_heuristic_rf_value"]


df["final_heuristic_rf_value"] = df.apply(apply_final_logic, axis=1)


In [108]:
df["error_prophet"] = df["pred_value_prophet"] - df["vol_in_rum_value"]
df["abs_error_prophet"] = df["error_prophet"].abs()

df["error_rf"] = df["pred_value_rf"] - df["vol_in_rum_value"]
df["abs_error_rf"] = df["error_rf"].abs()

df["error_final_heuristic_prophet"] = df["final_heuristic_prophet_value"] - df["vol_in_rum_value"]
df["abs_error_final_heuristic_prophet"] = df["error_final_heuristic_prophet"].abs()

df["error_final_heuristic_rf"] = df["final_heuristic_rf_value"] - df["vol_in_rum_value"]
df["abs_error_final_heuristic_rf"] = df["error_final_heuristic_rf"].abs()


In [109]:
df["recency_heuristic_rf"] = df.apply(
    lambda r: max(r["recency_factor"] * r["P3M"], r["pred_rf"]),
    axis=1
)

# Seasonality heuristic: max(seasonality_value, forecast)
df["seasonality_heuristic_rf"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY'], r["pred_rf"]),
    axis=1
)

# Recency + Seasonality combined
df["recency_seasonality_heuristic_rf"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY'] * r["recency_factor"],
                  r["pred_rf"]),
    axis=1
)

# Non-seasonal heuristic
df["non_seasonal_heuristic_rf"] = df["pred_rf"] * df["shrink_ratio_rf"]


# --------------------------------------------------
# 3. FINAL DECISION TREE + skipped condition
# --------------------------------------------------

def apply_final_logic(r):

    # Rule 1: skipped → force P3M
    if r.get("skipped", 0) == 1:
        return r["P3M_value"]

    
    # Rule 3: Heuristic combinations
    if r["final_trend"] == 1 and r["seasonality_flag"] == 0:
        return r["recency_heuristic_rf"]


    if r["final_trend"] != 1 and r["seasonality_flag"] == 1:
        return r["seasonality_heuristic_rf"]
    
    if r["final_trend"] == 1 and r["seasonality_flag"] == 1:
        return r["recency_seasonality_heuristic_rf"]

    # Default: (0,0)
    return r["non_seasonal_heuristic_rf"]


df["final_heuristic_rf"] = df.apply(apply_final_logic, axis=1)


In [110]:
df["recency_heuristic_prophet"] = df.apply(
    lambda r: max(r["recency_factor"] * r["P3M"], r["pred_prophet"]),
    axis=1
)

# Seasonality heuristic: max(seasonality_value, forecast)
df["seasonality_heuristic_prophet"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY'], r["pred_prophet"]),
    axis=1
)

# Recency + Seasonality combined
df["recency_seasonality_heuristic_prophet"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY'] * r["recency_factor"],
                  r["pred_prophet"]),
    axis=1
)

# Non-seasonal heuristic
df["non_seasonal_heuristic_prophet"] = df["pred_prophet"] * df["shrink_ratio_prophet"]


# --------------------------------------------------
# 3. FINAL DECISION TREE + skipped condition
# --------------------------------------------------

def apply_final_logic(r):

    # Rule 1: skipped → force P3M
    if r.get("skipped", 0) == 1:
        return r["P3M_value"]

    
    # Rule 3: Heuristic combinations
    if r["final_trend"] == 1 and r["seasonality_flag"] == 0:
        return r["recency_heuristic_prophet"]


    if r["final_trend"] != 1 and r["seasonality_flag"] == 1:
        return r["seasonality_heuristic_prophet"]
    
    if r["final_trend"] == 1 and r["seasonality_flag"] == 1:
        return r["recency_seasonality_heuristic_prophet"]

    # Default: (0,0)
    return r["non_seasonal_heuristic_prophet"]


df["final_heuristic_prophet"] = df.apply(apply_final_logic, axis=1)


In [112]:
df.to_csv('Heuristics_all_combination_ecom_cp.csv')

In [114]:
df[(df['M month'].notna())].to_csv('Heuristics_all_combination_ecom_cp2.csv')

In [2]:
import pandas as pd
final_df = pd.read_csv('Heuristics_all_combination_ecom_cp.csv')

/tmp/ipykernel_2052077/3783379672.py:2: DtypeWarning: Columns (20,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  final_df = pd.read_csv('Heuristics_all_combination_ecom_cp.csv')


In [6]:
import numpy as np
import pandas as pd

def compute_thresholds(df_grp):
    """
    df_grp MUST contain:
    - month_date
    - vol_in_rum
    - run_month

    Returns: lower_threshold, upper_threshold, mean, std
    """

    df_grp = df_grp.sort_values("month_date")
    run_month = df_grp["run_month"].max()

    # --- Use ONLY actual data (strictly before run month)
    df_actual = df_grp[df_grp["month_date"] < run_month]

    series = df_actual["vol_in_rum_value"].astype(float).values

    # If no real data → return zeros
    if len(series) == 0:
        return pd.Series({
            "lower_threshold": 0,
            "upper_threshold": 0,
            "mean_value": 0,
            "std_value": 0
        })

    # --- Take last 12 months OR all available
    if len(series) > 12:
        series = series[-12:]

    mean_val = np.mean(series)
    std_val = np.std(series)

    # --- SPECIAL CASE: ≤3 data points
    if len(series) <= 3:
        lower = 0.5 * mean_val
        upper = 2 * mean_val

        return pd.Series({
            "lower_threshold": lower,
            "upper_threshold": upper,
            "mean_value": mean_val,
            "std_value": std_val
        })

    # --- Normal case (std can be zero also)
    lower = max(0,mean_val - 2*std_val)
    upper = mean_val + 3*std_val

    return pd.Series({
        "lower_threshold": lower,
        "upper_threshold": upper,
        "mean_value": mean_val,
        "std_value": std_val
    })

threshold_df = final_df.groupby(
    ["platform_name", "parent_material_code", "run_month"]
).apply(compute_thresholds).reset_index()

threshold_df.head()


,platform_name,parent_material_code,run_month,lower_threshold,upper_threshold,mean_value,std_value
0,Amazon ARIPL,718288,2025-12-31,0.134916,0.345766,0.219256,0.042170
1,Amazon ARIPL,718321,2025-12-31,0.000000,0.000000,0.000000,0.000000
2,Amazon ARIPL,718322,2025-12-31,0.060035,0.108314,0.079347,0.009656
3,Amazon ARIPL,718323,2025-12-31,0.000000,0.000000,0.000000,0.000000
4,Amazon ARIPL,718328,2025-12-31,0.016588,0.032115,0.022799,0.003105


In [7]:
threshold_df.to_csv('ecom_threshold.csv')

In [162]:
final_df[(final_df['M month'].notna())].to_csv('Heuristics_all_combination_qcom_cp_chk.csv')

In [129]:
final_df[final_df['month_date'] == '2026-02-28']['LY P3M_value'].sum()

123.92211585241307

In [49]:
df['TREND'].unique()

array([ 1,  0, -1])

In [96]:
prophet_output = collate_file('prophet_data_train_till')

downloaded_results\new_pipeline\202508-09_FK_Others_Offtakes\train_till_30_Jun_2025\prophet_results\prophet_data_train_till_30_Jun_2025.csv
downloaded_results\new_pipeline\202508-09_FK_Others_Offtakes\train_till_31_Jul_2025\prophet_results\prophet_data_train_till_31_Jul_2025.csv
downloaded_results\new_pipeline\202509_AZ_BB_Offtakes\train_till_30_Jun_2025\prophet_results\prophet_data_train_till_30_Jun_2025.csv
downloaded_results\new_pipeline\202509_AZ_BB_Offtakes\train_till_31_Jul_2025\prophet_results\prophet_data_train_till_31_Jul_2025.csv


In [100]:
len_before_merge = len(offtake_df)
offtake_df = offtake_df.merge(
    read_qtr_ind_rate_table()[['brand_code', 'qtr_ind_rate']] ,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(offtake_df)


Credentials retrieved successfully for prod db.


In [ ]:
offtake_df

In [104]:
offtake_df['OT_Value_in_Cr'] = offtake_df['vol_in_rum'] * offtake_df['qtr_ind_rate'] / (10 ** 7)

In [105]:
offtake_df.to_csv('Offtake_realigned_base.csv', index=False)

In [108]:
trend_file_df[trend_file_df['run_month'].isin(['2025-10-31'])].to_csv('Trend_File_Oct_Live_Run_OT.csv', index=False)

In [107]:
trend_file_df[trend_file_df['run_month'].isin(['2025-07-31', '2025-08-31'])].to_csv('Trend_File_SepAug_OT.csv', index=False)

In [97]:
prophet_output.to_csv('ECOM_Prophet_Trend_OT_Chain_PSKU_AugSep.csv', index=False)

In [81]:
forecast_df = trend_file_df[trend_file_df['month_date'] >= trend_file_df['run_month']]

forecast_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,LY P6M,P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,LY,LLY,LY value,LLY value
26,Amazon_718589,2025-03-31,1312.900000,1293.400000,1824.634404,945.661643,0.059090,0.058213,0.082122,0.042562,...,841.200000,0.059090,0.058213,0.027878,0.037860,0.082122,717.3,1428.0,0.032284,0.064271
27,Amazon_718589,2025-04-30,1312.900000,1293.400000,1894.995393,1306.938750,0.059090,0.058213,0.085289,0.058822,...,792.800000,0.059090,0.058213,0.036843,0.035682,0.085289,1053.0,1040.4,0.047393,0.046826
28,Amazon_718589,2025-05-31,1312.900000,1293.400000,2055.487818,1194.591429,0.059090,0.058213,0.092512,0.053765,...,682.250000,0.059090,0.058213,0.039944,0.030706,0.092512,1241.1,1236.3,0.055859,0.055643
29,Amazon_718589,2025-06-30,1312.900000,1293.400000,1713.411024,989.591786,0.059090,0.058213,0.077116,0.044539,...,811.600000,0.059090,0.058213,0.045178,0.036528,0.077116,797.4,1176.0,0.035889,0.052929
56,Amazon_722188,2025-03-31,0.666667,6.400000,0.000000,471.682496,0.000030,0.000288,0.000000,0.021229,...,782.533333,0.000030,0.000288,0.043045,0.035220,0.000000,806.4,236.8,0.036294,0.010658
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120425,Flipkart National_809422,2025-10-31,211.833333,159.550000,262.584032,129.597491,0.030576,0.023029,0.037901,0.018706,...,207.383333,0.030576,0.023029,0.009040,0.029934,0.046132,0.8,NaN,0.000115,NaN
120442,Flipkart National_809423,2025-07-31,75.000000,53.533333,0.000000,169.044957,0.010826,0.007727,0.000000,0.024400,...,NaN,0.010826,0.007727,0.057582,NaN,0.008194,243.4,NaN,0.035132,NaN
120443,Flipkart National_809423,2025-08-31,75.000000,53.533333,0.000000,117.175693,0.010826,0.007727,0.000000,0.016913,...,NaN,0.010826,0.007727,0.054599,NaN,0.000000,99.6,NaN,0.014376,NaN
120444,Flipkart National_809423,2025-09-30,75.000000,53.533333,0.000000,111.920263,0.010826,0.007727,0.000000,0.016155,...,286.050000,0.010826,0.007727,0.048200,0.041288,0.000000,75.4,NaN,0.010883,NaN


In [98]:
trend_file_df[trend_file_df['run_month'] == '2025-08-31'].to_csv('Trend_file_Sep.csv', index=False)

In [82]:
forecast_df['is_na'] = forecast_df['vol_in_rum'].isna()
forecast_df.groupby(['run_month', 'month_date'])['is_na'].sum()

run_month   month_date
2025-03-31  2025-03-31    0
            2025-04-30    0
            2025-05-31    0
            2025-06-30    0
2025-04-30  2025-04-30    0
            2025-05-31    0
            2025-06-30    0
            2025-07-31    0
2025-05-31  2025-05-31    0
            2025-06-30    0
            2025-07-31    0
            2025-08-31    0
2025-06-30  2025-06-30    0
            2025-07-31    0
            2025-08-31    0
            2025-09-30    0
2025-07-31  2025-07-31    0
            2025-08-31    0
            2025-09-30    0
            2025-10-31    0
Name: is_na, dtype: int64

In [83]:
forecast_df.drop('is_na', axis=1, inplace=True)

In [84]:
pred_value_cols = [col for col in trend_file_df.columns if 'value' in col and 'pred' in col]
pred_value_cols

['pred_value_p3m',
 'pred_value_p6m',
 'pred_value_prophet',
 'pred_value_rf',
 'pred_value_best_model',
 'pred_prophet_70%ile_value']

In [85]:
for col in pred_value_cols:
    trend_file_df[f'error_{col}'] = trend_file_df[col] - trend_file_df['vol_in_rum_value']
    trend_file_df[f'abs_error_{col}'] = np.abs(trend_file_df[col] - trend_file_df['vol_in_rum_value'])    

In [86]:
trend_file_df.to_csv('Trend_file_OT_FK_AZ_BB.csv', index=False)

In [85]:
feature_importance_df = collate_file('feature_importance_train_till')

downloaded_results\202504-08_AZ_BB_ChainPSKU_OT_run\train_till_28_Feb_2025\ml_results\feature_importance_train_till_28_Feb_2025.csv
downloaded_results\202504-08_AZ_BB_ChainPSKU_OT_run\train_till_30_Apr_2025\ml_results\feature_importance_train_till_30_Apr_2025.csv
downloaded_results\202504-08_AZ_BB_ChainPSKU_OT_run\train_till_30_Jun_2025\ml_results\feature_importance_train_till_30_Jun_2025.csv
downloaded_results\202504-08_AZ_BB_ChainPSKU_OT_run\train_till_31_Mar_2025\ml_results\feature_importance_train_till_31_Mar_2025.csv
downloaded_results\202504-08_AZ_BB_ChainPSKU_OT_run\train_till_31_May_2025\ml_results\feature_importance_train_till_31_May_2025.csv


In [87]:
feature_importance_df.to_excel('202505-AZ_BB_OT_Feature_Imp.xlsx', index=False)

In [11]:
collate_file('prophet_data_train_till_').to_excel('202504-08_Prophet_File.xlsx', index=False)

downloaded_results\ECOM_ChainPSKU_OT_run\train_till_28_Feb_2025\prophet_results\prophet_data_train_till_28_Feb_2025.csv
downloaded_results\ECOM_ChainPSKU_OT_run\train_till_30_Apr_2025\prophet_results\prophet_data_train_till_30_Apr_2025.csv
downloaded_results\ECOM_ChainPSKU_OT_run\train_till_30_Jun_2025\prophet_results\prophet_data_train_till_30_Jun_2025.csv
downloaded_results\ECOM_ChainPSKU_OT_run\train_till_31_Mar_2025\prophet_results\prophet_data_train_till_31_Mar_2025.csv
downloaded_results\ECOM_ChainPSKU_OT_run\train_till_31_May_2025\prophet_results\prophet_data_train_till_31_May_2025.csv


### Secondary

In [87]:
sec_query = """SELECT 
    chain,
    parent_material_code, 
    month_date, 
    SUM(sec_actuals_vol_rum_month) AS sec_vol_actuals_rum_month,
    SUM(sec_apo_plan_vol_rum_month) AS sec_apo_plan_vol_rum_month
FROM (
    SELECT 
        month_date, 
        distributor_code, 
        material_code, 
        sec_actuals_vol_rum_month, 
        sec_apo_plan_vol_rum_month
    FROM 
        dwh_bpm_dist_brand_mth_sbp 
    WHERE 
        month_date BETWEEN '2022-04-01' AND '2025-12-31'
) A
JOIN (
    SELECT DISTINCT
        customer, 
        chain
    FROM 
        mst_chain_master 
    WHERE 
        chain_type = 'E Com B2C'
) CC
    ON A.distributor_Code = CC.customer
JOIN (
    SELECT 
        material_code, 
        parent_material_code 
    FROM 
        mst_material 
    WHERE 
        company_code = 'MIL' 
        AND latest_record_ind = 1
) M 
    ON A.material_code = M.material_code
GROUP BY 
    chain,
    parent_material_code, 
    month_date
ORDER BY 
    chain,
    parent_material_code, 
    month_date;
"""


results = pd.read_sql(con=prod_conn, sql=sec_query)
sales_data = pd.DataFrame(results)
sales_data.columns = sales_data.columns.str.lower()
sales_data = sales_data.rename(columns={'parent_material1_code':'parent_material_code'})

In [88]:
sales_data['month_date'] = pd.to_datetime(sales_data['month_date'])

In [89]:
sales_data['chain'] = sales_data['chain'].replace({
    'Grofers': 'Blinkit',
    'Amazon B2C': 'Amazon ARIPL',
    'Flipkart-Grocery': 'Flipkart Grocery',
    'FlipkartGrocery': 'Flipkart Grocery',
    'Flipkart-National': 'Flipkart National',
    'RK WORLDINFOCOM': 'Amazon RK',
    'ZEPTO': 'Zepto',
    'Kiranakart Technologies': 'Zepto',
    'Big basket B2B': 'Big Basket',
    'Big basket B2C': 'Big Basket',
    'Myntra': 'MYNTRA'
})

In [90]:
chains = ['Big Basket', 'Blinkit', 'Amazon ARIPL', 'Flipkart Grocery',
       'Flipkart National', 'Swiggy', 'Zepto', 'Amazon RK', 'City Mall',
       '1MG', 'Dealshare', 'Meesho', 'Nykaa', 'First Cry', 'MYNTRA',
       'Purplle']

In [91]:
for chain in chains:
    if chain not in sales_data['chain'].unique():
        print(chain)

In [92]:
dev_conn = get_dbconnection('DEV')
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment_2""",
    dev_conn
)

realignment_df.columns = realignment_df.columns.str.lower()

def realign_pskus(data, channel='ECOM'):
    realignment_data = realignment_df.copy()
    realignment_data = realignment_data[
        (realignment_data["channel"] == channel)
        | (realignment_data["channel"] == channel + " B2C")
        | (realignment_data["channel"] == "All")
    ]
    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']
    
    data["parent_material_code"] = data["parent_material_code"].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data['parent_material_code'] == old_psku, "parent_material_code"
        ] = new_psku

    return data


Credentials retrieved successfully for dev db.


In [93]:
realigned_df = realign_pskus(sales_data.copy())

In [94]:
realigned_df = realigned_df.groupby(
    ['chain', 'parent_material_code', 'month_date'], as_index=False
).sum()

In [95]:
old_pskus = realignment_df[
    (realignment_df["channel"] == "ECOM")
    | (realignment_df["channel"] == "ECOM" + " B2C")
    | (realignment_df["channel"] == "All")
]['psku old'].unique()

for psku in old_pskus:
    assert psku not in realigned_df['parent_material_code'].unique()

In [96]:
realigned_df['key'] = realigned_df['chain'] + '_' + realigned_df['parent_material_code'].astype(str) 
realigned_df['month_date'] = pd.to_datetime(realigned_df['month_date'])

realigned_df.duplicated(subset=['key', 'month_date']).sum()

0

In [97]:
realigned_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()

0

In [99]:
realigned_df


,chain,parent_material_code,month_date,sec_vol_actuals_rum_month,sec_apo_plan_vol_rum_month,key
0,1MG,715095,2024-01-31,0.0,0.000,1MG_715095
1,1MG,715096,2023-05-31,0.0,0.000,1MG_715096
2,1MG,715096,2023-06-30,0.0,0.189,1MG_715096
3,1MG,715096,2023-07-31,0.0,0.016,1MG_715096
4,1MG,715096,2023-08-31,0.0,0.000,1MG_715096
...,...,...,...,...,...,...
1081559,imli,807033,2025-04-30,0.0,0.000,imli_807033
1081560,imli,807033,2025-05-31,0.0,0.000,imli_807033
1081561,imli,807033,2025-06-30,0.0,0.000,imli_807033
1081562,imli,807033,2025-07-31,0.0,0.000,imli_807033


In [105]:
trend_file_df['platform_name'].unique()

array(['Amazon', 'Big Basket', 'Flipkart Grocery', 'Flipkart National'],
      dtype=object)

In [106]:
trend_file_df['platform_updated'] = np.where(
    trend_file_df['platform_name'] == 'Amazon', 
    np.where(
        trend_file_df['portfolio'].isin(['Foods', 'Saffola Oils']), 
        'Amazon ARIPL', 
        'Amazon RK'
    ),
    trend_file_df['platform_name']
)

In [108]:
trend_file_df['platform_updated'].unique()

array(['Amazon RK', 'Big Basket', 'Flipkart Grocery', 'Flipkart National',
       'Amazon ARIPL'], dtype=object)

In [109]:
trend_file_df.groupby(
    'platform_name'
)['platform_updated'].unique()

platform_name
Amazon               [Amazon RK, Amazon ARIPL]
Big Basket                        [Big Basket]
Flipkart Grocery            [Flipkart Grocery]
Flipkart National          [Flipkart National]
Name: platform_updated, dtype: object

In [102]:
trend_file_df['portfolio']

0             Hair Oils
1             Hair Oils
2             Hair Oils
3             Hair Oils
4             Hair Oils
              ...      
120441    Male Grooming
120442    Male Grooming
120443    Male Grooming
120444    Male Grooming
120445    Male Grooming
Name: portfolio, Length: 120446, dtype: object

In [113]:
trend_file_df['key'] = trend_file_df[
    ['platform_updated', 'parent_material_code']].astype(str).agg('_'.join, axis=1)

In [116]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    realigned_df[['key', 'month_date', 'sec_vol_actuals_rum_month']],
    on=['key', 'month_date'],
    how='left'
)
assert len_before_merge == len(trend_file_df)
del len_before_merge